# TRACE
## Temporal Regime-Aware Causal Ensemble with Guarded Dual-Strategy Selection

**Operational objective:** At each forecast origin $t$, use the preceding 168 hours to predict PV generation for all 24 horizons, $\hat{y}_{t+1}, \ldots, \hat{y}_{t+24}$.  
**Calendar-based split (exact 50/25/25):** January 2015–April 2017 for training (28 months), May 2017–June 2018 for validation (14 months), and July 2018–August 2019 for testing (14 months).  
**Primary metrics:** RMSE, MAE, and $R^2$ over every origin–horizon pair, including periods with zero observed generation.

TRACE does not impose the same zero-handling strategy on every site. It first compares two validation-only feature profiles: `full_context` and `short_memory`, which removes weekly-memory features. Within each profile, the notebook evaluates:

1. a **hurdle strategy** that combines a structural-night rule, an activity classifier ensemble, and a positive-magnitude regressor ensemble; and
2. a **direct-all strategy** that fixes structural nighttime at zero while fitting one regressor to both daylight zeros and positive PV values.

The direct-all strategy must pass a conservative Pareto guard: its validation RMSE must improve by at least 2% relative to the hurdle strategy, and its validation MAE must not worsen. After the profile and strategy are locked, all final experts are refitted on the combined training and validation periods. The test period is evaluated only after these choices are fixed.

The RNN, LSTM, GRU, BiLSTM, 1D-CNN, 1D-CNN-BiLSTM, Attn-GRU, TCN, Transformer, and TCFN benchmarks are retrained with the same direct 24-step target and the same test forecast origins.

> In this notebook, **causal** does not refer to causal-effect estimation. It means that the source time of every measured input is no later than the forecast origin $t$. Because the candidate design was refined after diagnosing earlier runs on the same test period, results on that period should be treated as developmental and exploratory. Confirmatory claims require a frozen-model evaluation on a later period or an external site.


## Quick Start

1. Place the two site CSV files in the repository root, `data/`, or `upload/`, or set `TRACE_DATA_DIR`.
2. Run the setup cell once. Uncomment the `%pip install` line only when dependencies are missing.
3. Use the default configuration for the full reproducible experiment.
4. Set `TRACE_FAST=1` only for a short structural smoke test.
5. Set `TRACE_SKIP_DEEP=1` only when preprocessing and tree-model checks are needed without deep benchmarks.

All generated tables, figures, predictions, statistical tests, and serialized model bundles are written to `TRACE_outputs/`.

> GitHub renders the notebook as a static document. Execute it locally, in Jupyter, or in Colab to reproduce the results.


## Why Zero-Generation Periods Are Retained

A practical PV forecasting system must correctly represent nighttime zeros, sunrise and sunset transitions, and zero output during cloudy daylight hours. Reporting RMSE, MAE, or $R^2$ only after removing rows with zero observations can therefore overstate operational performance. This notebook reports three complementary scopes:

1. **Overall — primary:** All direct 24-step origin–horizon pairs, including observed zeros. This is the primary result.
2. **Solar-eligible:** Pairs that are not classified as structural nighttime by the training-derived month–hour rule. Observed daylight zeros remain included.
3. **Actual-positive — diagnostic:** Only rows with observed PV generation above zero. This scope diagnoses positive-magnitude regression and is not the primary result.

The hurdle candidate uses an `ExtraTreesClassifier` and a `HistGradientBoostingClassifier`, together with two regression experts trained only on positive samples. The direct-all candidate uses a `HistGradientBoostingRegressor` trained jointly on daylight zeros and positive samples after structural nighttime is excluded. The feature profile, zero-handling strategy, expert weights, and activity threshold are selected exclusively from validation data; they are never adjusted after inspecting the test zero distribution.


In [ ]:
# Run this installation command once only if the required packages are missing.
# %pip install -U "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" scipy matplotlib seaborn joblib torch shap

from __future__ import annotations

import gc
import json
import os
import platform
import random
import time
import warnings

# Set this before importing PyTorch to support deterministic CUDA execution.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.environ.get("TEMP", "/tmp"), "trace_mpl"))
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from matplotlib.patches import FancyBboxPatch
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import norm, wilcoxon

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
FAST_MODE = os.environ.get("TRACE_FAST", "0") == "1"
SKIP_DEEP = os.environ.get("TRACE_SKIP_DEEP", "0") == "1"

@dataclass(frozen=True)
class Config:
    # Calendar-month split: 28/14/14 months, exactly 50/25/25.
    train_start: str = "2015-01-01 00:00:00"
    val_start: str = "2017-05-01 00:00:00"
    test_start: str = "2018-07-01 00:00:00"
    data_end_exclusive: str = "2019-09-01 00:00:00"
    history_hours: int = 168
    horizon: int = 24
    # Full mode evaluates every hourly test origin; only training uses stride=3 to reduce overlap.
    train_origin_stride: int = 24 if FAST_MODE else 3
    val_origin_stride: int = 6 if FAST_MODE else 1
    test_origin_stride: int = 6 if FAST_MODE else 1
    trials_per_expert: int = 1 if FAST_MODE else 5
    feature_profiles: tuple[str, ...] = ("full_context", "short_memory")
    direct_switch_rmse_margin: float = 0.02
    direct_switch_mae_tolerance: float = 0.00
    profile_complexity_tie_tolerance: float = 1e-6
    tree_jobs: int = 2 if FAST_MODE else max(1, min(6, os.cpu_count() or 1))
    selection_mae_weight: float = 0.25
    zero_epsilon: float = 0.0
    figure_dpi: int = 600
    run_xai: bool = False if FAST_MODE else True
    xai_rows: int = 1500
    bootstrap_repeats: int = 100 if FAST_MODE else 1000
    # Faithful PyTorch ports of the repository benchmarks. FAST_MODE runs a smaller smoke-test subset.
    run_deep_benchmarks: bool = not SKIP_DEEP
    deep_lookback: int = 24
    deep_max_epochs: int = 2 if FAST_MODE else 30
    deep_patience: int = 1 if FAST_MODE else 5
    deep_batch_size: int = 256
    deep_num_workers: int = 0  # Safe default for Windows and Jupyter
    deep_models: tuple[str, ...] = (
        ("GRU", "TCN", "TCFN") if FAST_MODE else
        ("RNN", "LSTM", "GRU", "BiLSTM", "1D-CNN", "1D-CNN-BiLSTM", "Attn-GRU", "TCN", "Transformer", "TCFN")
    )
    run_strong_tcfn_168: bool = False if FAST_MODE else True
    deep_repeats: int = 1 if FAST_MODE else 5
    run_retrained_ablation: bool = False if FAST_MODE else True
    output_dir: str = "TRACE_outputs"

CFG = Config()
OUTPUT_DIR = Path(CFG.output_dir).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SITE_LATITUDE = {"Dangjin": 36.9, "Gwangyang": 34.9}  # Approximate site latitude used by the solar-position proxy
DATASET_FILES = {
    "Dangjin": ["Dangjin_Landfill_PV_Dataset.csv", "Dangjin_Landfill_PV_Dataset(1).csv"],
    "Gwangyang": ["Gwangyang_Port_Site2_PV_Dataset.csv", "Gwangyang_Port_Site2_PV_Dataset(1).csv"],
}

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "fast_mode": FAST_MODE,
    "history_hours": CFG.history_hours,
    "forecast_horizon": CFG.horizon,
    "output_dir": str(OUTPUT_DIR),
})


## System Architecture

The model never uses weather observations from the forecast target interval. Because the datasets do not contain archived numerical weather prediction (NWP) forecasts with issue timestamps, the main experiment constructs a 24-hour weather proxy from observations available by time $t$, primarily the previous day's value at the corresponding target hour. In an operational deployment, archived site-specific NWP forecasts can replace or augment this proxy after retraining. Actual future weather from the test interval must not be substituted as if it were an operational forecast.


In [ ]:
def draw_trace_architecture(save=True):
    fig, ax = plt.subplots(figsize=(17, 9))
    ax.set_xlim(0, 17); ax.set_ylim(0, 10); ax.axis("off")

    def box(x, y, w, h, title, body, color):
        patch = FancyBboxPatch(
            (x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.12",
            facecolor=color, edgecolor="#263746", linewidth=1.35,
        )
        ax.add_patch(patch)
        ax.text(x+w/2, y+h*.70, title, ha="center", va="center",
                fontsize=10.3, fontweight="bold", color="#17202a")
        ax.text(x+w/2, y+h*.30, body, ha="center", va="center",
                fontsize=7.9, color="#34495e", linespacing=1.2)

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="-|>", lw=1.45, color="#34495e"))

    box(.2, 3.65, 2.15, 2.6, "Forecast origin t",
        "Observed PV + weather\nt−167 ... t\nNo source time > t", "#d6eaf8")
    box(2.65, 6.55, 2.35, 1.7, "Historical state",
        "levels, ramps, rolling statistics\nfull vs short-memory profile", "#e8f8f5")
    box(2.65, 4.15, 2.35, 1.7, "Known target context",
        "lead 1...24, calendar cycles\nsolar-elevation proxy", "#e8f8f5")
    box(2.65, 1.75, 2.35, 1.7, "Weather proxy",
        "previous-day target hour\ntrend adjustment from data ≤ t", "#e8f8f5")
    box(5.45, 3.75, 2.15, 2.5, "Causal feature matrix",
        "one row per lead h\nvalidation screens context profile", "#d6eaf8")
    box(8.05, 7.15, 2.35, 1.55, "Structural-zero gate",
        "train-learned month × hour", "#fadbd8")
    box(8.05, 4.25, 2.35, 2.25, "Hurdle strategy",
        "activity ensemble +\npositive-only magnitude ensemble", "#fcf3cf")
    box(8.05, 1.35, 2.35, 2.15, "Direct-all strategy",
        "one regressor learns daylight\nzeros and positive magnitudes", "#fdebd0")
    box(11.15, 3.55, 2.65, 3.0, "Validation-only Pareto guard",
        "profile comparison\ndirect requires ≥2% RMSE gain\nand non-inferior MAE\nthen train+validation refit", "#f5eef8")
    box(14.45, 4.05, 2.25, 2.25, "Direct 24-step output",
        "ŷ(t+1), ..., ŷ(t+24)\nupdated again at t+1", "#d5f5e3")

    for y2 in (7.4, 5.0, 2.6): arrow(2.35, 5.0, 2.65, y2)
    arrow(5.0, 7.4, 5.45, 5.75); arrow(5.0, 5.0, 5.45, 5.0); arrow(5.0, 2.6, 5.45, 4.25)
    arrow(7.6, 5.0, 8.05, 5.35); arrow(7.6, 4.65, 8.05, 2.4); arrow(7.6, 5.35, 8.05, 7.9)
    arrow(10.4, 7.9, 11.15, 6.05); arrow(10.4, 5.35, 11.15, 5.25); arrow(10.4, 2.4, 11.15, 4.3)
    arrow(13.8, 5.05, 14.45, 5.15)

    ax.text(8.5, 9.55, "TRACE: guarded dual-strategy rolling multi-step forecasting",
            ha="center", fontsize=15.5, fontweight="bold")
    ax.text(8.5, 9.05, "Every hour t, one selected strategy returns the full t+1 ... t+24 trajectory",
            ha="center", fontsize=9.8, color="#566573")
    fig.tight_layout()
    if save:
        path = OUTPUT_DIR / "TRACE_architecture.png"
        fig.savefig(path, dpi=CFG.figure_dpi, bbox_inches="tight", facecolor="white")
        print("saved:", path)
    plt.show()

draw_trace_architecture()


## 1. Data Loading, Timeline Reconstruction, and the 50/25/25 Split

The source codebook records `Hour=1...24`; this notebook converts those values to timestamp hours `0...23`. The 24-hour gap at Gwangyang on 16 April 2015 is not interpolated. Any forecast origin whose 168-hour history or 24-hour target window crosses that gap is removed.

For portable execution, place the required CSV files in the repository root, `data/`, or `upload/`. You may instead set the `TRACE_DATA_DIR` environment variable to an explicit dataset directory. The resolver also supports standard Colab and sandbox locations.

The complete 56-month period is divided by **calendar month** into 28 training months, 14 validation months, and 14 test months. Split membership is determined by the dates of **all 24 target timestamps**, not merely by the forecast origin. For example, the first test origin is 30 June 2018 at 23:00 because all 24 targets begin on 1 July 2018. Its inputs still consist only of information observed by the forecast origin.


In [ ]:
REQUIRED_COLUMNS = [
    "Year", "Month", "Day", "Hour", "AverageTemp", "LowTemp", "HighTemp",
    "RainFall", "SteamPress", "DewPoint", "Sunshine", "Insolation",
    "Cloudiness", "GroundTemp", "Temp", "Wind", "Press", "Humi", "Solar_Power",
]
# Exclude daily summary temperatures that may depend on observations later than the forecast origin.
OPERATIONAL_WEATHER = [
    "RainFall", "SteamPress", "DewPoint", "Sunshine", "Insolation",
    "Cloudiness", "GroundTemp", "Temp", "Wind", "Press", "Humi",
]
TYPE_A_TO_B = {
    "year": "Year", "month": "Month", "day": "Day", "hour": "Hour",
    "temp_mean": "AverageTemp", "temp_min": "LowTemp", "temp_max": "HighTemp",
    "precip": "RainFall", "vapor_pressure": "SteamPress", "dew_point": "DewPoint",
    "sunshine_duration": "Sunshine", "solar_radiation": "Insolation",
    "cloud_cover": "Cloudiness", "ground_temp": "GroundTemp", "air_temp": "Temp",
    "wind_speed": "Wind", "air_pressure": "Press", "relative_humidity": "Humi",
    "solar_power": "Solar_Power",
}

def resolve_file(candidates):
    """Resolve a dataset path across common local, GitHub, Colab, and sandbox locations."""
    roots = []

    # Optional explicit location, for example:
    #   Windows PowerShell: $env:TRACE_DATA_DIR = "C:\\path\\to\\data"
    #   Linux/macOS:       export TRACE_DATA_DIR="/path/to/data"
    configured_root = os.environ.get("TRACE_DATA_DIR")
    if configured_root:
        roots.append(Path(configured_root).expanduser())

    roots.extend([
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "upload",
        Path.home(),
        Path("/content"),
        Path("/mnt/data"),
    ])

    # Preserve order while removing duplicate paths.
    unique_roots = list(dict.fromkeys(root.resolve() for root in roots))
    for root in unique_roots:
        for filename in candidates:
            path = root / filename
            if path.exists():
                return path.resolve()

    searched = "\n".join(str(root / filename) for root in unique_roots for filename in candidates)
    raise FileNotFoundError(
        "None of the required dataset files was found. "
        "Place the CSV files in the repository root, data/, or upload/, "
        "or set TRACE_DATA_DIR. Searched paths:\n" + searched
    )

def load_site(path: Path, site: str) -> pd.DataFrame:
    raw = pd.read_csv(path).rename(columns=TYPE_A_TO_B)
    missing = sorted(set(REQUIRED_COLUMNS)-set(raw.columns))
    if missing:
        raise ValueError(f"{site}: missing {missing}")
    raw = raw[REQUIRED_COLUMNS].copy()
    if not raw.Hour.between(1, 24).all():
        raise ValueError(f"{site}: Hour must be 1..24")
    timestamp = (
        pd.to_datetime(dict(year=raw.Year, month=raw.Month, day=raw.Day), errors="raise")
        + pd.to_timedelta(raw.Hour.astype(int)-1, unit="h")
    )
    raw.index = pd.DatetimeIndex(timestamp, name="timestamp")
    raw = raw.sort_index()
    if raw.index.duplicated().any():
        raise ValueError(f"{site}: duplicated timestamps")
    full_index = pd.date_range(raw.index.min(), raw.index.max(), freq="h", name="timestamp")
    frame = raw.reindex(full_index)
    frame["observed_row"] = frame.Solar_Power.notna() & frame[OPERATIONAL_WEATHER].notna().all(axis=1)
    frame["source_hour"] = frame.index.hour + 1
    frame.attrs.update(site=site, source_path=str(path))
    return frame

DATA = {}
audit = []
for site, candidates in DATASET_FILES.items():
    path = resolve_file(candidates)
    frame = load_site(path, site)
    DATA[site] = frame
    audit.append({
        "site": site, "file": path.name, "complete_hours": len(frame),
        "observed_hours": int(frame.observed_row.sum()),
        "missing_hours": int((~frame.observed_row).sum()),
        "first": frame.index.min(), "last": frame.index.max(),
        "zero_pct": 100*frame.Solar_Power.eq(0).mean(), "max_power": frame.Solar_Power.max(),
    })
DATA_AUDIT = pd.DataFrame(audit)
display(DATA_AUDIT)

SPLIT_BOUNDS = {
    "train": (pd.Timestamp(CFG.train_start), pd.Timestamp(CFG.val_start)),
    "val": (pd.Timestamp(CFG.val_start), pd.Timestamp(CFG.test_start)),
    "test": (pd.Timestamp(CFG.test_start), pd.Timestamp(CFG.data_end_exclusive)),
}
month_rows = []
for split, (start, end) in SPLIT_BOUNDS.items():
    months = len(pd.period_range(start.to_period("M"), (end-pd.Timedelta(hours=1)).to_period("M"), freq="M"))
    month_rows.append({
        "split": split, "target_start": start, "target_end_inclusive": end-pd.Timedelta(hours=1),
        "calendar_months": months, "share_of_56_months_pct": 100*months/56,
    })
CALENDAR_SPLIT = pd.DataFrame(month_rows)
display(CALENDAR_SPLIT)
assert CALENDAR_SPLIT.calendar_months.tolist() == [28, 14, 14]
assert np.allclose(CALENDAR_SPLIT.share_of_56_months_pct, [50.0, 25.0, 25.0])


## 2. Direct 24-Step Feature Construction

Each forecast origin produces 24 horizon-aligned rows. The rows share the same historical state but have different `lead_hour`, target calendar variables, and solar-elevation proxies. The following target-aligned lag variables remain leakage-safe:

- `pv_target_lag24(h) = P(t+h-24)`, which is always observed by $t$ for $1 \le h \le 24$;
- `pv_target_lag168(h) = P(t+h-168)`, which is also always observed by $t$; and
- the weather proxy, defined as the previous day's observation at the target hour plus a decayed bias adjustment measured at the forecast origin.

Actual PV generation and actual weather for `t+1...t+24` are attached only as targets after the feature matrix has been completed. The model predicts all horizons directly and does not feed earlier predictions back into later horizons, thereby avoiding recursive error propagation.


In [ ]:
@dataclass
class MultiStepPack:
    site: str
    split: str
    X: pd.DataFrame
    y: pd.Series
    meta: pd.DataFrame
    origin_count: int

def solar_elevation_proxy(index: pd.DatetimeIndex, latitude_deg: float) -> np.ndarray:
    doy = index.dayofyear.to_numpy(dtype=float)
    local_mid_hour = index.hour.to_numpy(dtype=float)+0.5
    lat = np.deg2rad(latitude_deg)
    decl = np.deg2rad(23.44)*np.sin(2*np.pi*(284.0+doy)/365.2425)
    hour_angle = np.deg2rad(15.0*(local_mid_hour-12.5))
    sin_elev = np.sin(lat)*np.sin(decl)+np.cos(lat)*np.cos(decl)*np.cos(hour_angle)
    return np.clip(sin_elev, 0.0, 1.0)

def valid_origins(
    frame: pd.DataFrame, stride: int,
    target_start: pd.Timestamp, target_end_exclusive: pd.Timestamp,
) -> np.ndarray:
    n = len(frame)
    p = np.arange(CFG.history_hours-1, n-CFG.horizon, stride, dtype=int)
    history_ok = frame.observed_row.astype(np.int8).to_numpy()
    target_ok = frame.Solar_Power.notna().astype(np.int8).to_numpy()
    hist_csum = np.r_[0, np.cumsum(history_ok)]
    target_csum = np.r_[0, np.cumsum(target_ok)]
    start = p-CFG.history_hours+1
    end = p+CFG.horizon
    history_count = hist_csum[p+1]-hist_csum[start]
    target_count = target_csum[end+1]-target_csum[p+1]
    target_first = frame.index.take(p+1)
    target_last = frame.index.take(p+CFG.horizon)
    in_split = (target_first >= target_start) & (target_last < target_end_exclusive)
    return p[(history_count == CFG.history_hours) & (target_count == CFG.horizon) & in_split]

def assemble_origin_horizon_features(frame: pd.DataFrame, site: str, origins: np.ndarray):
    """Create 24 horizon-aligned feature rows per origin without reading future targets or weather."""
    H = CFG.horizon
    leads = np.arange(1, H+1, dtype=int)
    p = np.asarray(origins, dtype=int)
    target_pos_2d = p[:, None]+leads[None, :]
    target_pos = target_pos_2d.reshape(-1)
    origin_ts = frame.index.take(p)
    target_ts = frame.index.take(target_pos)
    power = frame.Solar_Power.to_numpy(dtype=float)

    def repeat_origin(values):
        return np.repeat(np.asarray(values), H)

    feat = {}

    # PV state observed at the forecast origin.
    for lag in (0, 1, 2, 3, 6, 12, 23, 24, 47, 48, 72, 167):
        feat[f"pv_origin_lag_{lag}"] = repeat_origin(power[p-lag])
    power_series = pd.Series(power, index=frame.index)
    for window in (3, 6, 12, 24, 72, 168):
        roll = power_series.rolling(window, min_periods=window)
        feat[f"pv_origin_mean_{window}"] = repeat_origin(roll.mean().to_numpy()[p])
        feat[f"pv_origin_std_{window}"] = repeat_origin(roll.std(ddof=0).to_numpy()[p])
    feat["pv_origin_ramp_1"] = repeat_origin(power[p]-power[p-1])
    feat["pv_origin_ramp_24"] = repeat_origin(power[p]-power[p-24])

    # Weather state observed at the origin; same-day summary variables are excluded.
    weather_arrays = {c: frame[c].to_numpy(dtype=float) for c in OPERATIONAL_WEATHER}
    for col, arr in weather_arrays.items():
        for lag in (0, 1, 2, 24, 167):
            feat[f"wx_origin_{col}_lag_{lag}"] = repeat_origin(arr[p-lag])
        feat[f"wx_origin_{col}_ramp_1"] = repeat_origin(arr[p]-arr[p-1])

    for col in ("Insolation", "Sunshine", "Cloudiness", "Temp", "Humi", "Wind"):
        series = pd.Series(weather_arrays[col], index=frame.index)
        for window in (6, 24, 168):
            roll = series.rolling(window, min_periods=window)
            feat[f"wx_origin_{col}_mean_{window}"] = repeat_origin(roll.mean().to_numpy()[p])
            feat[f"wx_origin_{col}_std_{window}"] = repeat_origin(roll.std(ddof=0).to_numpy()[p])

    # Information known in advance for each target timestamp.
    target_idx = pd.DatetimeIndex(target_ts)
    hour = target_idx.hour.to_numpy(dtype=float)
    doy = target_idx.dayofyear.to_numpy(dtype=float)
    dow = target_idx.dayofweek.to_numpy(dtype=float)
    lead_flat = np.tile(leads, len(p)).astype(float)
    for name, values, period in [
        ("target_hour", hour, 24.0), ("target_doy", doy, 365.2425),
        ("target_dow", dow, 7.0), ("lead", lead_flat, 24.0),
    ]:
        feat[f"{name}_sin"] = np.sin(2*np.pi*values/period)
        feat[f"{name}_cos"] = np.cos(2*np.pi*values/period)
    feat["lead_normalized"] = lead_flat/24.0
    elev = solar_elevation_proxy(target_idx, SITE_LATITUDE[site])
    feat["target_solar_elevation"] = elev
    feat["target_is_weekend"] = (dow >= 5).astype(float)

    # Horizon-aligned seasonal persistence; assert that every source timestamp is at or before the origin.
    day_idx = p[:, None]+leads[None, :]-24
    week_idx = p[:, None]+leads[None, :]-168
    assert np.all(day_idx <= p[:, None]) and np.all(week_idx <= p[:, None])
    feat["pv_target_lag24"] = power[day_idx].reshape(-1)
    feat["pv_target_lag168"] = power[week_idx].reshape(-1)

    # Use a leakage-safe previous-day same-hour proxy instead of actual future weather.
    decay = np.exp(-(leads.astype(float)-1.0)/24.0)[None, :]
    for col, arr in weather_arrays.items():
        aligned = arr[day_idx]
        daily_bias = (arr[p]-arr[p-24])[:, None]
        adjusted = aligned + decay*daily_bias
        if col in {"RainFall", "SteamPress", "Sunshine", "Insolation", "Cloudiness", "Wind", "Humi"}:
            adjusted = np.maximum(adjusted, 0.0)
        feat[f"wx_proxy_{col}_day_persistence"] = aligned.reshape(-1)
        feat[f"wx_proxy_{col}_bias_adjusted"] = adjusted.reshape(-1)

    feat["interaction_proxy_insolation_x_sun"] = feat["wx_proxy_Insolation_bias_adjusted"]*elev
    feat["interaction_proxy_sunshine_x_clear"] = (
        feat["wx_proxy_Sunshine_bias_adjusted"]
        * (1-np.clip(feat["wx_proxy_Cloudiness_bias_adjusted"], 0, 10)/10)
    )
    feat["interaction_proxy_temp_dew_spread"] = (
        feat["wx_proxy_Temp_bias_adjusted"]-feat["wx_proxy_DewPoint_bias_adjusted"]
    )

    X = pd.DataFrame(feat, dtype=np.float32)
    meta = pd.DataFrame({
        "forecast_origin": np.repeat(origin_ts.to_numpy(), H),
        "target_time": target_ts.to_numpy(),
        "horizon": np.tile(leads, len(p)),
        "target_source_hour": target_idx.hour+1,
        "target_month": target_idx.month,
        "target_solar_elevation": elev,
    })
    if not np.isfinite(X.to_numpy()).all():
        bad = X.columns[~np.isfinite(X.to_numpy()).all(axis=0)].tolist()
        raise ValueError(f"Non-finite features: {bad[:10]}")
    return X, meta, target_pos

def build_multistep_pack(
    frame: pd.DataFrame, site: str, split: str, stride: int,
    target_start: pd.Timestamp, target_end_exclusive: pd.Timestamp,
) -> MultiStepPack:
    origins = valid_origins(frame, stride, target_start, target_end_exclusive)
    X, meta, target_pos = assemble_origin_horizon_features(frame, site, origins)
    y = pd.Series(frame.Solar_Power.to_numpy(dtype=float)[target_pos], name="Solar_Power")
    assert y.notna().all() and len(y) == len(X) == len(meta)
    assert (meta.target_time > meta.forecast_origin).all()
    assert meta.horizon.between(1, 24).all()
    assert len(meta) == len(origins)*CFG.horizon
    assert meta.target_time.min() >= target_start
    assert meta.target_time.max() < target_end_exclusive
    return MultiStepPack(site, split, X, y, meta, len(origins))

def prepare_site_packs(frame: pd.DataFrame, site: str):
    specs = {
        "train": (SPLIT_BOUNDS["train"], CFG.train_origin_stride),
        "val": (SPLIT_BOUNDS["val"], CFG.val_origin_stride),
        "test": (SPLIT_BOUNDS["test"], CFG.test_origin_stride),
    }
    packs = {}
    for split, ((start, end), stride) in specs.items():
        packs[split] = build_multistep_pack(frame, site, split, stride, start, end)
    cols = packs["train"].X.columns
    assert cols.equals(packs["val"].X.columns) and cols.equals(packs["test"].X.columns)
    return packs

PACKS = {site: prepare_site_packs(frame, site) for site, frame in DATA.items()}
split_rows = []
for site, packs in PACKS.items():
    for split, pack in packs.items():
        split_rows.append({
            "site": site, "split": split, "origins": pack.origin_count,
            "origin_horizon_pairs": len(pack.y), "features": pack.X.shape[1],
            "first_origin": pack.meta.forecast_origin.min(),
            "last_origin": pack.meta.forecast_origin.max(),
            "first_target": pack.meta.target_time.min(),
            "last_target": pack.meta.target_time.max(),
        })
SPLIT_AUDIT = pd.DataFrame(split_rows)
display(SPLIT_AUDIT)
if not FAST_MODE:
    # Calendar-month lengths yield 10,224 validation hours and 10,248 test hours.
    assert PACKS["Dangjin"]["val"].origin_count == 10201
    assert PACKS["Dangjin"]["test"].origin_count == 10225
    assert PACKS["Gwangyang"]["val"].origin_count == 10201
    assert PACKS["Gwangyang"]["test"].origin_count == 10225
print("Causal multi-step audit passed.")


## 3. Separating Structural and Stochastic Zeros

Structural nighttime is defined from the training data: a `(target_month, target_source_hour)` combination is considered structural zero when its maximum observed PV generation is zero. This calendar rule is available before the test period begins.

For non-structural periods, TRACE evaluates two candidate strategies. The hurdle strategy first predicts whether PV generation is active and then estimates its positive magnitude. The direct-all strategy includes both cloudy-day zeros and positive values in one regression target. The hurdle formulation can be advantageous when activity classification is stable, whereas direct-all can be safer when small binary-gate errors would otherwise suppress meaningful PV output.


In [ ]:
def regression_metrics(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float).reshape(-1)
    yp = np.asarray(y_pred, dtype=float).reshape(-1)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "MAE": float(mean_absolute_error(yt, yp)),
        "R2": float(r2_score(yt, yp)) if np.unique(yt).size > 1 else np.nan,
    }

def selection_score(m):
    w = CFG.selection_mae_weight
    return (1-w)*m["RMSE"] + w*m["MAE"]

def structural_key(meta: pd.DataFrame) -> np.ndarray:
    return meta.target_month.to_numpy(dtype=int)*100 + meta.target_source_hour.to_numpy(dtype=int)

def learn_structural_gate(pack: MultiStepPack):
    tab = pd.DataFrame({"key": structural_key(pack.meta), "y": pack.y.to_numpy()})
    grouped = tab.groupby("key").y.agg(["count", "max"])
    keys = grouped.index[(grouped["count"] >= 30) & grouped["max"].le(CFG.zero_epsilon)].tolist()
    return {"structural_zero_keys": [int(k) for k in keys]}

def structural_zero_mask(meta: pd.DataFrame, state) -> np.ndarray:
    return np.isin(structural_key(meta), state["structural_zero_keys"])

def positive_probability(model, X):
    proba = model.predict_proba(X)
    classes = np.asarray(model.classes_)
    if 1 not in classes:
        return np.zeros(len(X), dtype=float)
    return proba[:, int(np.flatnonzero(classes == 1)[0])]

def make_model(kind: str, params: dict[str, Any], seed: int):
    if kind == "extra_cls":
        return ExtraTreesClassifier(
            n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
            min_samples_leaf=int(params["min_samples_leaf"]),
            min_samples_split=int(params["min_samples_split"]), max_features=float(params["max_features"]),
            class_weight="balanced_subsample", bootstrap=False, random_state=seed, n_jobs=CFG.tree_jobs,
        )
    if kind == "hist_cls":
        return HistGradientBoostingClassifier(
            learning_rate=float(params["learning_rate"]), max_iter=int(params["max_iter"]),
            max_leaf_nodes=int(params["max_leaf_nodes"]), min_samples_leaf=int(params["min_samples_leaf"]),
            l2_regularization=float(params["l2_regularization"]), max_bins=int(params["max_bins"]),
            early_stopping=False, random_state=seed,
        )
    if kind == "extra_reg":
        return ExtraTreesRegressor(
            n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
            min_samples_leaf=int(params["min_samples_leaf"]),
            min_samples_split=int(params["min_samples_split"]), max_features=float(params["max_features"]),
            bootstrap=False, random_state=seed, n_jobs=CFG.tree_jobs,
        )
    if kind == "hist_reg":
        return HistGradientBoostingRegressor(
            loss="squared_error", learning_rate=float(params["learning_rate"]),
            max_iter=int(params["max_iter"]), max_leaf_nodes=int(params["max_leaf_nodes"]),
            min_samples_leaf=int(params["min_samples_leaf"]),
            l2_regularization=float(params["l2_regularization"]), max_bins=int(params["max_bins"]),
            early_stopping=False, random_state=seed,
        )
    raise ValueError(kind)

def candidate_params(kind: str, n: int, seed: int):
    rng = np.random.default_rng(seed)
    if kind.startswith("extra"):
        base = {
            "n_estimators": 24 if FAST_MODE else 160, "max_depth": 12 if FAST_MODE else 20,
            "min_samples_leaf": 5, "min_samples_split": 4, "max_features": 0.70,
        }
        out = [base]
        while len(out) < n:
            out.append({
                "n_estimators": int(rng.choice([20, 32] if FAST_MODE else [120, 160, 220])),
                "max_depth": int(rng.choice([10, 14] if FAST_MODE else [14, 18, 24])),
                "min_samples_leaf": int(rng.integers(2, 16)),
                "min_samples_split": int(rng.integers(2, 15)),
                "max_features": float(rng.uniform(0.45, 0.90)),
            })
        return out
    base = {
        "learning_rate": 0.06, "max_iter": 20 if FAST_MODE else 250,
        "max_leaf_nodes": 15 if FAST_MODE else 31, "min_samples_leaf": 40,
        "l2_regularization": 1.0, "max_bins": 63 if FAST_MODE else 255,
    }
    out = [base]
    while len(out) < n:
        out.append({
            "learning_rate": float(np.exp(rng.uniform(np.log(0.025), np.log(0.12)))),
            "max_iter": int(rng.choice([15, 25] if FAST_MODE else [160, 240, 340])),
            "max_leaf_nodes": int(rng.choice([7, 15] if FAST_MODE else [15, 31, 63])),
            "min_samples_leaf": int(rng.integers(20, 101)),
            "l2_regularization": float(np.exp(rng.uniform(np.log(1e-3), np.log(20.0)))),
            "max_bins": int(rng.choice([31, 63] if FAST_MODE else [127, 255])),
        })
    return out

def training_subset(pack: MultiStepPack, gate_state, task: str):
    structural = structural_zero_mask(pack.meta, gate_state)
    active = pack.y.to_numpy() > CFG.zero_epsilon
    if task == "cls":
        keep = ~structural
        weights = compute_sample_weight(class_weight="balanced", y=active[keep].astype(int))
        return pack.X.loc[keep], active[keep].astype(int), weights
    keep = (~structural) & active
    return pack.X.loc[keep], pack.y.to_numpy()[keep], None

def tune_expert(site: str, packs, kind: str, gate_state):
    task = "cls" if kind.endswith("cls") else "reg"
    Xtr, ytr, weights = training_subset(packs["train"], gate_state, task)
    val = packs["val"]
    val_structural = structural_zero_mask(val.meta, gate_state)
    val_active = val.y.to_numpy() > CFG.zero_epsilon
    candidates = candidate_params(kind, CFG.trials_per_expert, SEED+sum(map(ord, kind)))
    best, records = None, []

    for trial, params in enumerate(candidates):
        started = time.perf_counter()
        model = make_model(kind, params, SEED+trial)
        if task == "cls":
            model.fit(Xtr, ytr, sample_weight=weights)
            pred_all = positive_probability(model, val.X)
            eval_mask = ~val_structural
            score = brier_score_loss(val_active[eval_mask].astype(int), pred_all[eval_mask])
            record = {
                "Brier": float(score),
                "F1_at_0.5": f1_score(val_active[eval_mask], pred_all[eval_mask] >= 0.5, zero_division=0),
                "selection_score": float(score),
            }
        else:
            model.fit(Xtr, ytr)
            pred_all = np.maximum(model.predict(val.X), 0.0)
            eval_mask = (~val_structural) & val_active
            m = regression_metrics(val.y.to_numpy()[eval_mask], pred_all[eval_mask])
            record = {**m, "selection_score": selection_score(m)}
        record.update({"site": site, "expert": kind, "trial": trial,
                       "fit_seconds": time.perf_counter()-started, **params})
        records.append(record)
        if best is None or record["selection_score"] < best["record"]["selection_score"]:
            best = {"model": model, "params": params.copy(), "prediction": pred_all.copy(), "record": record.copy()}
        else:
            del model
        gc.collect()
        print(f"{site:10s} {kind:9s} {trial+1}/{len(candidates)} score={record['selection_score']:.5f}")
    history = pd.DataFrame(records).sort_values("selection_score").reset_index(drop=True)
    history.to_csv(OUTPUT_DIR/f"{site}_{kind}_validation_search.csv", index=False)
    return best, history


## 4. Validation-Locked Context Profile and Guarded Dual Strategy

TRACE first compares `full_context` with `short_memory`, which removes the 167- and 168-hour weekly-memory features. For each profile, validation data determine the hurdle classifier and regressor ensemble weights and the activity-probability threshold. A separate direct-all regressor is evaluated on the same validation set.

Direct-all passes the Pareto guard only when its validation RMSE is at least 2% lower than the hurdle RMSE and its validation MAE is no worse. Otherwise, TRACE retains the hurdle strategy rather than switching zero-handling behavior for a marginal RMSE difference.

After the feature profile and strategy are locked, all final experts are refitted on the combined January 2015–June 2018 training and validation data. The July 2018–August 2019 test period does not influence selection. Under the hurdle strategy, the activity probability is used only for the zero/positive decision and is not multiplied into the predicted active magnitude.


In [ ]:
@dataclass
class Result:
    site: str
    models: dict[str, Any]
    params: dict[str, dict[str, Any]]
    decision: dict[str, Any]
    train_gate: dict[str, Any]
    final_gate: dict[str, Any]
    validation_metrics: dict[str, float]
    predictions: pd.DataFrame
    test_features: pd.DataFrame
    search_histories: dict[str, pd.DataFrame]
    candidate_table: pd.DataFrame
    refit_seconds: float


def apply_hurdle(prob, magnitude, meta, gate_state, decision):
    structural = structural_zero_mask(meta, gate_state)
    active = prob >= decision["threshold"]
    if decision["mode"] == "hard":
        pred = np.where(active, magnitude, 0.0)
    elif decision["mode"] == "magnitude_bypass":
        pred = magnitude.copy()
    else:
        raise ValueError(decision["mode"])
    pred = np.maximum(pred, 0.0)
    pred[structural] = 0.0
    return pred


def score_candidate(pack, pred):
    overall = regression_metrics(pack.y, pred)
    month_key = pd.to_datetime(pack.meta.target_time).dt.to_period("M")
    monthly = []
    for month in month_key.unique():
        mask = (month_key == month).to_numpy()
        monthly.append(selection_score(regression_metrics(pack.y.to_numpy()[mask], pred[mask])))
    monthly = np.asarray(monthly, dtype=float)
    robust = selection_score(overall) + 0.15 * monthly.std(ddof=1)
    return overall, monthly, float(robust)


def select_decision(val_pack, gate_state, expert_preds):
    rows = []
    for w_cls in (0.0, 0.25, 0.5, 0.75, 1.0):
        prob = w_cls*expert_preds["extra_cls"] + (1-w_cls)*expert_preds["hist_cls"]
        for w_reg in (0.0, 0.25, 0.5, 0.75, 1.0):
            mag = w_reg*expert_preds["extra_reg"] + (1-w_reg)*expert_preds["hist_reg"]
            for threshold in (0.25, 0.35, 0.45, 0.55, 0.65):
                decision = {
                    "weight_extra_classifier": w_cls, "weight_extra_regressor": w_reg,
                    "threshold": threshold, "mode": "hard",
                }
                pred = apply_hurdle(prob, mag, val_pack.meta, gate_state, decision)
                metrics, monthly, robust = score_candidate(val_pack, pred)
                rows.append({
                    **decision, **metrics, "selection_score": selection_score(metrics),
                    "monthly_score_mean": monthly.mean(), "monthly_score_std": monthly.std(ddof=1),
                    "robust_selection_score": robust,
                })
            decision = {
                "weight_extra_classifier": w_cls, "weight_extra_regressor": w_reg,
                "threshold": 0.5, "mode": "magnitude_bypass",
            }
            pred = apply_hurdle(prob, mag, val_pack.meta, gate_state, decision)
            metrics, monthly, robust = score_candidate(val_pack, pred)
            rows.append({
                **decision, **metrics, "selection_score": selection_score(metrics),
                "monthly_score_mean": monthly.mean(), "monthly_score_std": monthly.std(ddof=1),
                "robust_selection_score": robust,
            })
    table = pd.DataFrame(rows).sort_values(
        ["robust_selection_score", "selection_score", "mode"]
    ).reset_index(drop=True)
    best = table.iloc[0][[
        "weight_extra_classifier", "weight_extra_regressor", "threshold", "mode"
    ]].to_dict()
    for key in ("weight_extra_classifier", "weight_extra_regressor", "threshold"):
        best[key] = float(best[key])
    return best, table


def feature_profile_columns(columns, profile):
    columns = list(columns)
    if profile == "full_context":
        return columns
    if profile == "short_memory":
        return [c for c in columns if not ("168" in c or c.endswith("_167"))]
    raise ValueError(profile)


def subset_pack(pack, columns, split=None):
    return MultiStepPack(
        pack.site, split or pack.split, pack.X.loc[:, columns].copy(), pack.y.copy(),
        pack.meta.copy(), pack.origin_count,
    )


def make_profile_packs(packs, profile):
    cols = feature_profile_columns(packs["train"].X.columns, profile)
    return {split: subset_pack(pack, cols) for split, pack in packs.items()}


def predict_experts(models, X):
    return {
        "extra_cls": positive_probability(models["extra_cls"], X),
        "hist_cls": positive_probability(models["hist_cls"], X),
        "extra_reg": np.maximum(models["extra_reg"].predict(X), 0.0),
        "hist_reg": np.maximum(models["hist_reg"].predict(X), 0.0),
    }


def fit_final_expert(kind, params, final_pack, gate_state, seed):
    task = "cls" if kind.endswith("cls") else "reg"
    X, y, weights = training_subset(final_pack, gate_state, task)
    model = make_model(kind, params, seed)
    if task == "cls":
        model.fit(X, y, sample_weight=weights)
    else:
        model.fit(X, y)
    return model


def fit_fixed_experts(pack, gate_state, params, seed_offset):
    models = {}
    for i, kind in enumerate(("extra_cls", "hist_cls", "extra_reg", "hist_reg")):
        models[kind] = fit_final_expert(kind, params[kind], pack, gate_state, SEED+seed_offset+i)
    return models


def fit_direct_all(pack, gate_state, hist_params, seed):
    keep = ~structural_zero_mask(pack.meta, gate_state)
    model = make_model("hist_reg", hist_params, seed)
    model.fit(pack.X.loc[keep], pack.y.to_numpy()[keep])
    return model


def predict_direct_all(model, X, meta, gate_state):
    pred = np.maximum(model.predict(X), 0.0)
    pred[structural_zero_mask(meta, gate_state)] = 0.0
    return pred


def hurdle_from_experts(expert, meta, gate_state, decision):
    prob = (
        decision["weight_extra_classifier"]*expert["extra_cls"]
        +(1-decision["weight_extra_classifier"])*expert["hist_cls"]
    )
    magnitude = (
        decision["weight_extra_regressor"]*expert["extra_reg"]
        +(1-decision["weight_extra_regressor"])*expert["hist_reg"]
    )
    pred = apply_hurdle(prob, magnitude, meta, gate_state, decision)
    return prob, magnitude, pred


def choose_guarded_strategy(val_pack, hurdle_pred, direct_pred):
    hurdle_metrics, hurdle_monthly, hurdle_robust = score_candidate(val_pack, hurdle_pred)
    direct_metrics, direct_monthly, direct_robust = score_candidate(val_pack, direct_pred)
    direct_dominates = (
        direct_metrics["RMSE"] <= (1.0-CFG.direct_switch_rmse_margin)*hurdle_metrics["RMSE"]
        and direct_metrics["MAE"] <= (1.0+CFG.direct_switch_mae_tolerance)*hurdle_metrics["MAE"]
    )
    strategy = "direct_all" if direct_dominates else "hurdle"
    selected_metrics = direct_metrics if direct_dominates else hurdle_metrics
    selected_robust = direct_robust if direct_dominates else hurdle_robust
    table = pd.DataFrame([
        {"strategy": "hurdle", **hurdle_metrics, "monthly_score_std": hurdle_monthly.std(ddof=1),
         "robust_selection_score": hurdle_robust, "selected_by_guard": strategy == "hurdle"},
        {"strategy": "direct_all", **direct_metrics, "monthly_score_std": direct_monthly.std(ddof=1),
         "robust_selection_score": direct_robust, "selected_by_guard": strategy == "direct_all"},
    ])
    return strategy, selected_metrics, selected_robust, bool(direct_dominates), table


def predict_all_components(models, X, meta, gate_state, decision):
    expert = predict_experts(models, X)
    prob, magnitude, hurdle = hurdle_from_experts(expert, meta, gate_state, decision)
    direct = predict_direct_all(models["direct_reg"], X, meta, gate_state)
    selected = direct if decision["strategy"] == "direct_all" else hurdle
    return {
        **expert, "probability": prob, "magnitude": magnitude,
        "hurdle": hurdle, "direct_all": direct, "selected": selected,
    }


def combine_train_validation(site, train, val):
    return MultiStepPack(
        site, "train+val", pd.concat([train.X, val.X], ignore_index=True),
        pd.concat([train.y, val.y], ignore_index=True),
        pd.concat([train.meta, val.meta], ignore_index=True),
        train.origin_count+val.origin_count,
    )


def run_site(site: str, packs) -> Result:
    print("\n"+"="*100+f"\n{site}: TRACE validation-only profile and strategy selection")
    full_gate = learn_structural_gate(packs["train"])
    tuned, histories = {}, {}
    for kind in ("extra_cls", "hist_cls", "extra_reg", "hist_reg"):
        tuned[kind], histories[kind] = tune_expert(site, packs, kind, full_gate)
    params = {kind: tuned[kind]["params"] for kind in tuned}

    profile_records = {}
    candidate_rows = []
    for profile in CFG.feature_profiles:
        pp = make_profile_packs(packs, profile)
        gate = learn_structural_gate(pp["train"])
        if profile == "full_context":
            hurdle_models = {kind: tuned[kind]["model"] for kind in tuned}
            val_expert = {kind: tuned[kind]["prediction"] for kind in tuned}
        else:
            hurdle_models = fit_fixed_experts(pp["train"], gate, params, 2000)
            val_expert = predict_experts(hurdle_models, pp["val"].X)
        hurdle_decision, hurdle_table = select_decision(pp["val"], gate, val_expert)
        prob, magnitude, hurdle_pred = hurdle_from_experts(
            val_expert, pp["val"].meta, gate, hurdle_decision,
        )
        direct_model = fit_direct_all(pp["train"], gate, params["hist_reg"], SEED+6100)
        direct_pred = predict_direct_all(direct_model, pp["val"].X, pp["val"].meta, gate)
        strategy, val_metrics, robust, dominates, strategy_table = choose_guarded_strategy(
            pp["val"], hurdle_pred, direct_pred,
        )
        strategy_table.insert(0, "feature_profile", profile)
        strategy_table["direct_guard_passed"] = dominates
        candidate_rows.append(strategy_table)
        profile_records[profile] = {
            "packs": pp, "gate": gate, "strategy": strategy,
            "hurdle_decision": hurdle_decision, "validation_metrics": val_metrics,
            "robust": robust, "n_features": len(pp["train"].X.columns),
        }
        hurdle_table.to_csv(OUTPUT_DIR/f"{site}_{profile}_validation_hurdle_decision.csv", index=False)
        del hurdle_models, direct_model
        gc.collect()

    selected_profile = min(
        profile_records,
        key=lambda name: (profile_records[name]["robust"], profile_records[name]["n_features"]),
    )
    selected = profile_records[selected_profile]
    selected_packs = selected["packs"]
    train, val, test = selected_packs["train"], selected_packs["val"], selected_packs["test"]
    final_pack = combine_train_validation(site, train, val)
    final_gate = learn_structural_gate(final_pack)

    started = time.perf_counter()
    models = fit_fixed_experts(final_pack, final_gate, params, 3000)
    models["direct_reg"] = fit_direct_all(final_pack, final_gate, params["hist_reg"], SEED+6200)
    params["direct_reg"] = params["hist_reg"].copy()
    decision = selected["hurdle_decision"].copy()
    decision.update({
        "strategy": selected["strategy"], "feature_profile": selected_profile,
        "direct_switch_rmse_margin": CFG.direct_switch_rmse_margin,
        "direct_switch_mae_tolerance": CFG.direct_switch_mae_tolerance,
    })
    components = predict_all_components(models, test.X, test.meta, final_gate, decision)
    refit_seconds = time.perf_counter()-started

    out = test.meta.copy()
    out["actual"] = test.y.to_numpy()
    out["active_probability"] = components["probability"]
    out["predicted_positive_magnitude"] = components["magnitude"]
    out["pred_hurdle"] = components["hurdle"]
    out["pred_direct_all"] = components["direct_all"]
    out["pred_TRACE"] = components["selected"]
    out["actual_active"] = out.actual > CFG.zero_epsilon
    out["predicted_active"] = out.active_probability >= decision["threshold"]
    out["structural_zero"] = structural_zero_mask(test.meta, final_gate)
    out["pred_extra_cls"] = components["extra_cls"]
    out["pred_hist_cls"] = components["hist_cls"]
    out["pred_extra_reg"] = components["extra_reg"]
    out["pred_hist_reg"] = components["hist_reg"]

    candidate_table = pd.concat(candidate_rows, ignore_index=True)
    candidate_table["selected_profile"] = candidate_table.feature_profile.eq(selected_profile)
    candidate_table.to_csv(OUTPUT_DIR/f"{site}_validation_profile_strategy_audit.csv", index=False)
    print(candidate_table[[
        "feature_profile", "strategy", "RMSE", "MAE", "robust_selection_score",
        "direct_guard_passed", "selected_by_guard", "selected_profile",
    ]])
    print("selected:", {"feature_profile": selected_profile, "strategy": decision["strategy"], **decision})
    print("validation overall:", selected["validation_metrics"])
    print("FINAL TEST overall:", regression_metrics(out.actual, out.pred_TRACE))
    return Result(
        site, models, params, decision, full_gate, final_gate,
        selected["validation_metrics"], out, test.X.copy(), histories,
        candidate_table, refit_seconds,
    )


RESULTS = {site: run_site(site, packs) for site, packs in PACKS.items()}


## 5. Multi-Step Performance Tables

`Overall` is the primary evaluation scope. `Solar-eligible` contains origin–horizon pairs that are not structural nighttime under the month–hour rule, while retaining observed zeros within those periods. `Actual-positive` is a diagnostic for positive-magnitude regression. The notebook stores metrics for every horizon and also reports the mean and standard deviation across the 24 horizons.


In [ ]:
def scoped_metric_rows(result: Result):
    p = result.predictions
    masks = {
        "Overall (primary; zeros included)": np.ones(len(p), dtype=bool),
        "Solar-eligible (zeros included)": ~p.structural_zero.to_numpy(),
        "Actual-positive (diagnostic)": p.actual_active.to_numpy(),
        "Actual-zero (RMSE/MAE only)": ~p.actual_active.to_numpy(),
    }
    rows = []
    for scope, mask in masks.items():
        m = regression_metrics(p.actual.to_numpy()[mask], p.pred_TRACE.to_numpy()[mask])
        if "Actual-zero" in scope:
            m["R2"] = np.nan
        rows.append({"site": result.site, "scope": scope, "pairs": int(mask.sum()), **m})
    return rows

def activity_metrics(result: Result):
    p = result.predictions
    mask = ~p.structural_zero.to_numpy()
    yt = p.actual_active.to_numpy()[mask]
    yp = p.predicted_active.to_numpy()[mask]
    prob = p.active_probability.to_numpy()[mask]
    return {
        "site": result.site, "eligible_pairs": int(mask.sum()),
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "F1": f1_score(yt, yp, zero_division=0),
        "AUROC": roc_auc_score(yt, prob) if np.unique(yt).size > 1 else np.nan,
        "Brier": brier_score_loss(yt.astype(int), prob),
    }

SCOPE_METRICS = pd.DataFrame([row for r in RESULTS.values() for row in scoped_metric_rows(r)])
ACTIVITY_METRICS = pd.DataFrame([activity_metrics(r) for r in RESULTS.values()])
display(SCOPE_METRICS)
display(ACTIVITY_METRICS)

horizon_rows = []
for site, result in RESULTS.items():
    for h, g in result.predictions.groupby("horizon", sort=True):
        horizon_rows.append({"site": site, "horizon": int(h), "pairs": len(g),
                             **regression_metrics(g.actual, g.pred_TRACE)})
HORIZON_METRICS = pd.DataFrame(horizon_rows)
HORIZON_SUMMARY = HORIZON_METRICS.groupby("site").agg(
    RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    R2_mean=("R2", "mean"), R2_std=("R2", "std"),
).reset_index()
display(HORIZON_METRICS)
display(HORIZON_SUMMARY)

# Compute 24-hour energy and peak forecasts as supplementary operational metrics.
trajectory_rows = []
for site, result in RESULTS.items():
    grouped = result.predictions.groupby("forecast_origin", sort=False)
    actual_energy = grouped.actual.sum()
    pred_energy = grouped.pred_TRACE.sum()
    actual_peak = grouped.actual.max()
    pred_peak = grouped.pred_TRACE.max()
    trajectory_rows.append({
        "site": site,
        "origins": len(actual_energy),
        "daily_energy_RMSE": np.sqrt(mean_squared_error(actual_energy, pred_energy)),
        "daily_energy_MAE": mean_absolute_error(actual_energy, pred_energy),
        "peak_RMSE": np.sqrt(mean_squared_error(actual_peak, pred_peak)),
        "peak_MAE": mean_absolute_error(actual_peak, pred_peak),
    })
TRAJECTORY_METRICS = pd.DataFrame(trajectory_rows)
display(TRAJECTORY_METRICS)

for name, table in {
    "scope_metrics": SCOPE_METRICS, "activity_metrics": ACTIVITY_METRICS,
    "horizon_metrics": HORIZON_METRICS, "horizon_summary": HORIZON_SUMMARY,
    "trajectory_metrics": TRAJECTORY_METRICS,
}.items():
    table.to_csv(OUTPUT_DIR/f"TRACE_{name}.csv", index=False)


## 6. Fair Direct 24-Step Benchmarks Based on the Official TCFN Repository

The official [`johnnyone89/TCFN4PVForecasting`](https://github.com/johnnyone89/TCFN4PVForecasting) repository was reviewed at commit `bdb5aa1c6fd9cfe1bb84704930567201e074cf94`. It provides:

- `main_tcfn_pipeline.ipynb`: the TCFN architecture with a Conv1D trend branch, multi-head-attention context branch, and LSTM fusion;
- `benchmark_comparison.ipynb`: RNN, LSTM, GRU, BiLSTM, 1D-CNN, 1D-CNN-BiLSTM, Attn-GRU, TCN, and Transformer benchmarks; and
- `ablation_study_variants.ipynb`: ablations of convolution, attention, LSTM, and cyclical encoding.

The original implementation uses a 24-hour history to predict **one next time step**. The reproduction below preserves the layer order and common `Deep_Robust` settings—128 units, dropout 0.5, Adam with a learning rate of 0.0005, MSE loss, and up to 30 epochs—while replacing the final `Dense(1)` output with a direct `Dense(24)` output. Every model uses the same causal inputs, 24 target horizons, and hourly test origins without actual future weather. Four TCFN settings from the original study are compared on validation data, and `TCFN-168` provides a stronger sensitivity benchmark with the same 168-hour history length as TRACE.

> The reproduction is implemented in PyTorch, but its computational graphs correspond to the TensorFlow/Keras structures in the source repository. The original one-step values in TCFN Table 3 and the direct 24-step results below represent different forecasting tasks and are therefore reported separately.


In [ ]:
DEEP_RUN_METRICS = pd.DataFrame()
DEEP_HORIZON_METRICS = pd.DataFrame()
DEEP_HORIZON_SUMMARY = pd.DataFrame()
DEEP_PREDICTIONS = {site: {} for site in DATA}
DEEP_SELECTION = pd.DataFrame()

if CFG.run_deep_benchmarks:
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        from torch.utils.data import DataLoader, TensorDataset
        from sklearn.preprocessing import StandardScaler
    except ImportError as exc:
        raise ImportError(
            "PyTorch benchmarks are enabled. Run the `%pip install ... torch` command in the setup cell and "
            "restart the kernel. Use TRACE_SKIP_DEEP=1 only for preprocessing smoke tests."
        ) from exc

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print({
        "torch": torch.__version__, "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "benchmark_models": CFG.deep_models,
        "training_repeats": CFG.deep_repeats,
    })

    def seed_everything(seed: int):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass

    BENCHMARK_RAW_COLUMNS = ["Solar_Power", *OPERATIONAL_WEATHER]
    BENCHMARK_FEATURE_NAMES = [
        *BENCHMARK_RAW_COLUMNS, "hour_sin", "hour_cos", "doy_sin", "doy_cos",
    ]

    def benchmark_raw_frame(frame: pd.DataFrame) -> pd.DataFrame:
        idx = frame.index
        out = frame[BENCHMARK_RAW_COLUMNS].copy()
        out["hour_sin"] = np.sin(2*np.pi*idx.hour/24.0)
        out["hour_cos"] = np.cos(2*np.pi*idx.hour/24.0)
        out["doy_sin"] = np.sin(2*np.pi*idx.dayofyear/365.2425)
        out["doy_cos"] = np.cos(2*np.pi*idx.dayofyear/365.2425)
        return out.astype(np.float32)

    def fit_sequence_scaler(frame: pd.DataFrame, end_exclusive: pd.Timestamp):
        raw = benchmark_raw_frame(frame)
        keep = (raw.index < end_exclusive) & frame.observed_row
        scaler = StandardScaler().fit(raw.loc[keep, BENCHMARK_FEATURE_NAMES].to_numpy(dtype=np.float32))
        return scaler

    def origin_positions(frame: pd.DataFrame, pack: MultiStepPack) -> np.ndarray:
        origins = pd.DatetimeIndex(pack.meta.forecast_origin.iloc[::CFG.horizon])
        pos = frame.index.get_indexer(origins)
        if (pos < 0).any() or len(pos) != pack.origin_count:
            raise AssertionError("origin alignment failed")
        return pos

    def sequence_arrays(
        frame: pd.DataFrame, pack: MultiStepPack, lookback: int, scaler: StandardScaler,
    ):
        raw = benchmark_raw_frame(frame)
        pos = origin_positions(frame, pack)
        offsets = np.arange(-lookback+1, 1, dtype=int)
        row_index = pos[:, None] + offsets[None, :]
        values = raw.to_numpy(dtype=np.float32)[row_index]
        if not np.isfinite(values).all():
            raise ValueError(f"{pack.site}/{pack.split}: non-finite deep input")
        shape = values.shape
        values = scaler.transform(values.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)
        y = pack.y.to_numpy(dtype=np.float32).reshape(pack.origin_count, CFG.horizon)
        meta = pack.meta.copy().reset_index(drop=True)
        return values, y, meta

    class CausalConv1d(nn.Module):
        def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
            super().__init__()
            self.trim = dilation*(kernel_size-1)
            self.conv = nn.Conv1d(
                in_channels, out_channels, kernel_size,
                padding=self.trim, dilation=dilation,
            )
        def forward(self, x):
            y = self.conv(x)
            return y[..., :-self.trim] if self.trim else y

    class KerasStyleMHA(nn.Module):
        """Match the projection dimensions of Keras MultiHeadAttention(num_heads, key_dim)."""
        def __init__(self, input_dim, heads, key_dim):
            super().__init__()
            self.heads, self.key_dim = heads, key_dim
            inner = heads*key_dim
            self.q = nn.Linear(input_dim, inner)
            self.k = nn.Linear(input_dim, inner)
            self.v = nn.Linear(input_dim, inner)
            self.out = nn.Linear(inner, input_dim)
        def forward(self, query, key, value, need_weights=False):
            B, Tq, _ = query.shape; Tk = key.shape[1]
            q = self.q(query).view(B, Tq, self.heads, self.key_dim).transpose(1, 2)
            k = self.k(key).view(B, Tk, self.heads, self.key_dim).transpose(1, 2)
            v = self.v(value).view(B, Tk, self.heads, self.key_dim).transpose(1, 2)
            weights = torch.softmax(torch.matmul(q, k.transpose(-2, -1))/np.sqrt(self.key_dim), dim=-1)
            z = torch.matmul(weights, v).transpose(1, 2).contiguous().view(B, Tq, -1)
            out = self.out(z)
            return (out, weights) if need_weights else out

    class ResidualTCNBlock(nn.Module):
        def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
            super().__init__()
            self.conv = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
            self.norm = nn.BatchNorm1d(out_channels)
            self.drop = nn.Dropout(dropout)
            self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        def forward(self, x):
            return self.skip(x) + self.drop(F.silu(self.norm(self.conv(x))))

    class RepoDirectModel(nn.Module):
        """Direct 24-output PyTorch port of the benchmark architecture from the official repository."""
        def __init__(self, name: str, input_dim: int, lookback: int, units=128, dropout=.5, heads=4):
            super().__init__()
            self.name = name
            self.lookback = lookback
            self.dropout = nn.Dropout(dropout)
            if name == "RNN":
                self.seq = nn.RNN(input_dim, units, batch_first=True, nonlinearity="tanh")
                latent = units
            elif name == "LSTM":
                self.seq = nn.LSTM(input_dim, units, batch_first=True)
                latent = units
            elif name == "GRU":
                self.seq = nn.GRU(input_dim, units, batch_first=True)
                latent = units
            elif name == "BiLSTM":
                self.seq = nn.LSTM(input_dim, units, batch_first=True, bidirectional=True)
                latent = 2*units
            elif name == "1D-CNN":
                self.conv = nn.Sequential(
                    nn.Conv1d(input_dim, units, 3, padding=1), nn.BatchNorm1d(units),
                    nn.SiLU(), nn.MaxPool1d(2), nn.Flatten(), nn.Dropout(dropout),
                )
                latent = units*(lookback//2)
            elif name == "1D-CNN-BiLSTM":
                self.conv = nn.Sequential(
                    nn.Conv1d(input_dim, units, 3, padding=1), nn.BatchNorm1d(units),
                    nn.SiLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
                )
                self.seq = nn.LSTM(units, units, batch_first=True, bidirectional=True)
                latent = 2*units
            elif name == "Attn-GRU":
                self.seq = nn.GRU(input_dim, units, batch_first=True)
                self.attn = KerasStyleMHA(units, heads, units)
                self.norm = nn.LayerNorm(units)
                latent = units
            elif name == "TCN":
                blocks = []
                channels = input_dim
                for dilation in (1, 2, 4):
                    blocks.append(ResidualTCNBlock(channels, units, 3, dilation, .1))
                    channels = units
                self.tcn = nn.Sequential(*blocks)
                latent = units
            elif name == "Transformer":
                self.context_attn = KerasStyleMHA(input_dim, heads, units)
                self.context_norm1 = nn.LayerNorm(input_dim)
                self.context_ff = nn.Sequential(
                    nn.Linear(input_dim, units), nn.SiLU(), nn.Linear(units, input_dim),
                )
                self.context_norm2 = nn.LayerNorm(input_dim)
                latent = input_dim
            elif name == "TCFN":
                self.trend_conv = nn.Conv1d(input_dim, units, 3, padding=1)
                self.trend_bn = nn.BatchNorm1d(units)
                self.context_attn = KerasStyleMHA(input_dim, heads, 16)
                self.context_norm1 = nn.LayerNorm(input_dim)
                self.context_ff = nn.Linear(input_dim, input_dim)
                self.context_norm2 = nn.LayerNorm(input_dim)
                self.fusion_lstm = nn.LSTM(units+input_dim, units, batch_first=True)
                latent = units
            else:
                raise ValueError(name)
            self.head = nn.Sequential(
                nn.Linear(latent, units), nn.SiLU(), nn.Linear(units, CFG.horizon),
            )

        def encode(self, x, return_attention=False):
            attention = None
            if self.name in {"RNN", "LSTM", "GRU", "BiLSTM"}:
                z, _ = self.seq(x); z = z[:, -1]
            elif self.name == "1D-CNN":
                z = self.conv(x.transpose(1, 2))
            elif self.name == "1D-CNN-BiLSTM":
                z = self.conv(x.transpose(1, 2)).transpose(1, 2)
                z, _ = self.seq(z); z = z[:, -1]
            elif self.name == "Attn-GRU":
                z, _ = self.seq(x)
                a, attention = self.attn(z, z, z, need_weights=True)
                z = self.norm(z+a).mean(dim=1)
            elif self.name == "TCN":
                z = self.tcn(x.transpose(1, 2)).mean(dim=-1)
            elif self.name == "Transformer":
                a, attention = self.context_attn(
                    x, x, x, need_weights=True,
                )
                z = self.context_norm1(x+a)
                z = self.context_norm2(z+self.context_ff(z)).mean(dim=1)
            elif self.name == "TCFN":
                trend = self.dropout(F.silu(self.trend_bn(self.trend_conv(x.transpose(1, 2))))).transpose(1, 2)
                context_attn, attention = self.context_attn(
                    x, x, x, need_weights=True,
                )
                context = self.context_norm1(x+context_attn)
                context = self.context_norm2(context+F.silu(self.context_ff(context)))
                z, _ = self.fusion_lstm(torch.cat([trend, context], dim=-1))
                z = z[:, -1]
            return (z, attention) if return_attention else z

        def forward(self, x, return_attention=False):
            if return_attention:
                z, attention = self.encode(x, True)
                return torch.relu(self.head(self.dropout(z))), attention
            return torch.relu(self.head(self.dropout(self.encode(x))))

    def make_loader(X, y, shuffle: bool, seed: int):
        generator = torch.Generator().manual_seed(seed)
        batch_size = min(CFG.deep_batch_size, 64) if X.shape[1] >= 168 else CFG.deep_batch_size
        return DataLoader(
            TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
            batch_size=batch_size, shuffle=shuffle, generator=generator,
            num_workers=CFG.deep_num_workers, pin_memory=(DEVICE.type == "cuda"),
        )

    @torch.no_grad()
    def torch_predict(model, X, target_scale):
        model.eval(); chunks = []
        batch_size = min(CFG.deep_batch_size, 64) if X.shape[1] >= 168 else CFG.deep_batch_size
        loader = DataLoader(
            TensorDataset(torch.from_numpy(X)), batch_size=batch_size,
            shuffle=False, num_workers=CFG.deep_num_workers,
            pin_memory=(DEVICE.type == "cuda"),
        )
        for (xb,) in loader:
            chunks.append(model(xb.to(DEVICE, non_blocking=True)).cpu().numpy())
        return np.maximum(np.concatenate(chunks, axis=0)*target_scale, 0.0)

    def fit_validation_model(name, Xtr, ytr, Xva, yva, params, seed):
        seed_everything(seed)
        scale = max(float(np.nanpercentile(ytr, 99.9)), 1.0)
        ytr_s = (ytr/scale).astype(np.float32)
        model = RepoDirectModel(
            name, Xtr.shape[-1], Xtr.shape[1], units=params["units"],
            dropout=params["dropout"], heads=params.get("heads", 4),
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
        criterion = nn.MSELoss()
        loader = make_loader(Xtr, ytr_s, True, seed)
        best = {"score": np.inf, "epoch": 0, "metrics": None}
        stale = 0
        for epoch in range(1, CFG.deep_max_epochs+1):
            model.train()
            for xb, yb in loader:
                xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(xb), yb)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            pred = torch_predict(model, Xva, scale)
            m = regression_metrics(yva, pred)
            score = selection_score(m)
            if score < best["score"]-1e-8:
                best = {"score": score, "epoch": epoch, "metrics": m}
                stale = 0
            else:
                stale += 1
            if stale >= CFG.deep_patience:
                break
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return best

    def refit_and_predict(name, Xfit, yfit, Xte, params, epochs, seed):
        seed_everything(seed)
        scale = max(float(np.nanpercentile(yfit, 99.9)), 1.0)
        yfit_s = (yfit/scale).astype(np.float32)
        model = RepoDirectModel(
            name, Xfit.shape[-1], Xfit.shape[1], units=params["units"],
            dropout=params["dropout"], heads=params.get("heads", 4),
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
        criterion = nn.MSELoss()
        loader = make_loader(Xfit, yfit_s, True, seed)
        for _ in range(max(1, int(epochs))):
            model.train()
            for xb, yb in loader:
                xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(xb), yb)
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        pred = torch_predict(model, Xte, scale)
        return model, pred

    REPO_DEEP_ROBUST = {"units": 128, "dropout": .5, "lr": 5e-4, "heads": 4}
    TCFN_SCENARIOS = [
        {"id": "Light_Fast", "units": 32, "dropout": .2, "lr": 1e-3, "heads": 4},
        {"id": "Standard_Balanced", "units": 64, "dropout": .3, "lr": 5e-4, "heads": 4},
        {"id": "Deep_Slow", "units": 128, "dropout": .3, "lr": 1e-4, "heads": 4},
        {"id": "Deep_Robust", "units": 128, "dropout": .5, "lr": 5e-4, "heads": 4},
    ] if not FAST_MODE else [
        {"id": "smoke", "units": 16, "dropout": .2, "lr": 1e-3, "heads": 4},
    ]
else:
    print("Deep benchmark skipped only because TRACE_SKIP_DEEP=1.")


In [ ]:
if CFG.run_deep_benchmarks:
    selection_rows, run_rows, horizon_rows = [], [], []
    REPRESENTATIVE_DEEP_MODELS = {}

    for site, frame in DATA.items():
        print("\n"+"="*100+f"\n{site}: repository-derived direct 24-step benchmarks")
        packs = PACKS[site]
        scaler_train = fit_sequence_scaler(frame, pd.Timestamp(CFG.val_start))
        scaler_final = fit_sequence_scaler(frame, pd.Timestamp(CFG.test_start))

        array_cache = {}
        def arrays(split, lookback, final_scaler=False):
            key = (split, lookback, final_scaler)
            if key not in array_cache:
                scaler = scaler_final if final_scaler else scaler_train
                array_cache[key] = sequence_arrays(frame, packs[split], lookback, scaler)
            return array_cache[key]

        jobs = [(name, CFG.deep_lookback, name) for name in CFG.deep_models]
        if CFG.run_strong_tcfn_168:
            jobs.append(("TCFN", CFG.history_hours, "TCFN-168"))

        for architecture, lookback, label in jobs:
            Xtr, ytr, _ = arrays("train", lookback, False)
            Xva, yva, _ = arrays("val", lookback, False)

            if architecture == "TCFN":
                candidates = TCFN_SCENARIOS
            else:
                candidates = [{"id": "Repo_Deep_Robust", **REPO_DEEP_ROBUST}]

            candidate_results = []
            for ci, params in enumerate(candidates):
                started = time.perf_counter()
                selected = fit_validation_model(
                    architecture, Xtr, ytr, Xva, yva, params, SEED+100*ci,
                )
                row = {
                    "site": site, "model": label, "scenario": params["id"],
                    "lookback": lookback, "best_epoch": selected["epoch"],
                    "validation_selection_score": selected["score"],
                    "validation_RMSE": selected["metrics"]["RMSE"],
                    "validation_MAE": selected["metrics"]["MAE"],
                    "validation_R2": selected["metrics"]["R2"],
                    "selection_seconds": time.perf_counter()-started,
                }
                selection_rows.append(row)
                candidate_results.append((selected, params.copy()))
                print(label, params["id"], row)

            selected, params = min(candidate_results, key=lambda item: item[0]["score"])
            Xtr_f, ytr_f, _ = arrays("train", lookback, True)
            Xva_f, yva_f, _ = arrays("val", lookback, True)
            Xte, yte, meta_te = arrays("test", lookback, True)
            Xfit = np.concatenate([Xtr_f, Xva_f], axis=0)
            yfit = np.concatenate([ytr_f, yva_f], axis=0)
            repeat_predictions = []

            for repeat in range(CFG.deep_repeats):
                seed = SEED+1000*repeat+sum(map(ord, label))
                started = time.perf_counter()
                model, pred = refit_and_predict(
                    architecture, Xfit, yfit, Xte, params, selected["epoch"], seed,
                )
                repeat_predictions.append(pred)
                m = regression_metrics(yte, pred)
                run_rows.append({
                    "site": site, "model": label, "repeat": repeat, "seed": seed,
                    "lookback": lookback, "best_epoch": selected["epoch"],
                    "RMSE": m["RMSE"], "MAE": m["MAE"], "R2": m["R2"],
                    "refit_seconds": time.perf_counter()-started,
                })
                if repeat == 0:
                    REPRESENTATIVE_DEEP_MODELS[(site, label)] = {
                        "model": model.to("cpu"), "scaler": scaler_final,
                        "params": params, "epoch": selected["epoch"],
                        "lookback": lookback, "feature_names": BENCHMARK_FEATURE_NAMES,
                    }
                else:
                    del model
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

            # Averaging predictions from predefined seeds does not use the test set for model selection.
            ensemble_pred = np.mean(repeat_predictions, axis=0)
            flat = ensemble_pred.reshape(-1)
            DEEP_PREDICTIONS[site][label] = flat
            for h in range(1, CFG.horizon+1):
                hm = regression_metrics(yte[:, h-1], ensemble_pred[:, h-1])
                horizon_rows.append({"site": site, "model": label, "horizon": h, **hm})

            pred_frame = meta_te.copy()
            pred_frame["actual"] = yte.reshape(-1)
            pred_frame["prediction"] = flat
            pred_frame.to_csv(OUTPUT_DIR/f"{site}_{label.replace('/', '_')}_test_predictions.csv", index=False)

    DEEP_SELECTION = pd.DataFrame(selection_rows)
    DEEP_RUN_METRICS = pd.DataFrame(run_rows)
    DEEP_HORIZON_METRICS = pd.DataFrame(horizon_rows)
    DEEP_HORIZON_SUMMARY = DEEP_HORIZON_METRICS.groupby(["site", "model"]).agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"), R2_std=("R2", "std"),
    ).reset_index()
    DEEP_REPEAT_SUMMARY = DEEP_RUN_METRICS.groupby(["site", "model"]).agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"), R2_std=("R2", "std"),
        repeats=("repeat", "count"),
    ).reset_index()
    display(DEEP_SELECTION.sort_values(["site", "model", "validation_selection_score"]))
    display(DEEP_REPEAT_SUMMARY.sort_values(["site", "RMSE_mean"]))
    display(DEEP_HORIZON_SUMMARY.sort_values(["site", "RMSE_mean"]))
    DEEP_SELECTION.to_csv(OUTPUT_DIR/"repository_models_validation_selection.csv", index=False)
    DEEP_RUN_METRICS.to_csv(OUTPUT_DIR/"repository_models_repeat_metrics.csv", index=False)
    DEEP_REPEAT_SUMMARY.to_csv(OUTPUT_DIR/"repository_models_repeat_summary.csv", index=False)
    DEEP_HORIZON_METRICS.to_csv(OUTPUT_DIR/"repository_models_horizon_metrics.csv", index=False)
    DEEP_HORIZON_SUMMARY.to_csv(OUTPUT_DIR/"repository_models_horizon_summary.csv", index=False)
    for (site, label), bundle in REPRESENTATIVE_DEEP_MODELS.items():
        safe = label.replace("/", "_")
        torch.save({
            "architecture": "TCFN" if label == "TCFN-168" else label,
            "label": label, "lookback": bundle["lookback"],
            "feature_names": bundle["feature_names"], "params": bundle["params"],
            "epoch": bundle["epoch"], "state_dict": bundle["model"].state_dict(),
        }, OUTPUT_DIR/f"{site}_{safe}_representative_seed_checkpoint.pt")
        joblib.dump(bundle["scaler"], OUTPUT_DIR/f"{site}_{safe}_scaler.joblib")
else:
    DEEP_REPEAT_SUMMARY = pd.DataFrame()


## 7. Fair Benchmark Table and External One-Step Reference

TCFN Table 3 reports a **one-step-ahead** prediction from a 24-hour input window. Ranking TRACE's average performance across all 24 horizons against those one-step values would be inappropriate because the forecast horizons differ. This notebook therefore separates two comparisons:

- **Primary fair table:** Rolling direct 24-step results for all retrained models on July 2018–August 2019, including every horizon and every observed zero.
- **External reference table:** The original one-step values and TRACE's `h=1` slice. This table is descriptive only and is excluded from paired significance testing.


In [ ]:
PAIR_PREDICTIONS = {}
fair_overall_rows, fair_horizon_rows = [], []
for site, result in RESULTS.items():
    p = result.predictions.reset_index(drop=True)
    persistence168_source = (
        result.test_features if "pv_target_lag168" in result.test_features.columns
        else PACKS[site]["test"].X
    )
    preds = {
        "TRACE": p.pred_TRACE.to_numpy(),
        "Persistence-24": np.maximum(result.test_features["pv_target_lag24"].to_numpy(), 0.0),
        "Persistence-168": np.maximum(persistence168_source["pv_target_lag168"].to_numpy(), 0.0),
        **DEEP_PREDICTIONS.get(site, {}),
    }
    PAIR_PREDICTIONS[site] = preds
    actual = p.actual.to_numpy()
    for model, pred in preds.items():
        fair_overall_rows.append({"site": site, "model": model, **regression_metrics(actual, pred)})
        for h in range(1, CFG.horizon+1):
            mask = p.horizon.to_numpy() == h
            fair_horizon_rows.append({
                "site": site, "model": model, "horizon": h,
                **regression_metrics(actual[mask], np.asarray(pred)[mask]),
            })
    wide = p[["forecast_origin", "target_time", "horizon", "actual"]].copy()
    for model, pred in preds.items():
        wide[model] = pred
    wide.to_csv(OUTPUT_DIR/f"{site}_all_models_paired_test_predictions.csv", index=False)

FAIR_BENCHMARK_OVERALL = pd.DataFrame(fair_overall_rows)
FAIR_BENCHMARK_HORIZON = pd.DataFrame(fair_horizon_rows)
FAIR_BENCHMARK_HORIZON_SUMMARY = FAIR_BENCHMARK_HORIZON.groupby(["site", "model"]).agg(
    RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    R2_mean=("R2", "mean"), R2_std=("R2", "std"),
).reset_index()
FAIR_BENCHMARK_OVERALL["RMSE_rank"] = FAIR_BENCHMARK_OVERALL.groupby("site").RMSE.rank(method="min")
display(FAIR_BENCHMARK_OVERALL.sort_values(["site", "RMSE"]))
display(FAIR_BENCHMARK_HORIZON_SUMMARY.sort_values(["site", "RMSE_mean"]))
for site in DATA:
    tab = FAIR_BENCHMARK_OVERALL[FAIR_BENCHMARK_OVERALL.site == site].sort_values("RMSE")
    fig, axes = plt.subplots(1, 2, figsize=(18, 6.2))
    colors = ["#c0392b" if m == "TRACE" else "#7f8c8d" for m in tab.model]
    axes[0].barh(tab.model, tab.RMSE, color=colors)
    axes[0].invert_yaxis(); axes[0].set(xlabel="Overall RMSE (kW)", title="All-zero-inclusive direct 24-step ranking")
    selected_models = [m for m in ("TRACE", "TCFN", "TCFN-168", "TCN", "Persistence-24") if m in tab.model.values]
    for model in selected_models:
        hm = FAIR_BENCHMARK_HORIZON[
            (FAIR_BENCHMARK_HORIZON.site == site) & (FAIR_BENCHMARK_HORIZON.model == model)
        ]
        axes[1].plot(hm.horizon, hm.RMSE, marker="o", ms=3, lw=1.4, label=model)
    axes[1].set(xlabel="Forecast horizon", ylabel="RMSE (kW)", title="Error by horizon")
    axes[1].set_xticks(range(1, 25)); axes[1].legend(fontsize=8)
    fig.suptitle(f"{site}: fair rolling direct 24-step benchmark", fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f"{site}_fair_direct24_benchmark.png", dpi=CFG.figure_dpi,
                bbox_inches="tight", facecolor="white")
    plt.show()
FAIR_BENCHMARK_OVERALL.to_csv(OUTPUT_DIR/"fair_direct24_benchmark_overall.csv", index=False)
FAIR_BENCHMARK_HORIZON.to_csv(OUTPUT_DIR/"fair_direct24_benchmark_by_horizon.csv", index=False)
FAIR_BENCHMARK_HORIZON_SUMMARY.to_csv(
    OUTPUT_DIR/"fair_direct24_benchmark_horizon_mean_std.csv", index=False,
)

ONE_STEP_REFERENCE = {
    "Dangjin": {
        "TCFN paper (one-step)": (86.37, 39.49, 0.883),
        "AT-ResTCN attached run (one-step)": (67.3483, 34.3057, 0.9287),
    },
    "Gwangyang": {
        "TCFN paper (one-step)": (89.60, 43.07, 0.954),
        "AT-ResTCN attached run (one-step)": (87.5839, 42.9356, 0.9556),
    },
}
comparison_rows = []
for site, refs in ONE_STEP_REFERENCE.items():
    for model, (rmse, mae, r2) in refs.items():
        comparison_rows.append({
            "site": site, "model": model, "horizon_scope": "one-step external reference",
            "RMSE": rmse, "MAE": mae, "R2": r2,
        })
    one = HORIZON_METRICS[(HORIZON_METRICS.site == site) & (HORIZON_METRICS.horizon == 1)].iloc[0]
    overall = SCOPE_METRICS[(SCOPE_METRICS.site == site) & SCOPE_METRICS.scope.str.startswith("Overall")].iloc[0]
    comparison_rows.extend([
        {"site": site, "model": "TRACE h=1 slice", "horizon_scope": "h=1 from 24-step model",
         "RMSE": one.RMSE, "MAE": one.MAE, "R2": one.R2},
        {"site": site, "model": "TRACE operational", "horizon_scope": "all h=1...24",
         "RMSE": overall.RMSE, "MAE": overall.MAE, "R2": overall.R2},
    ])
EXTERNAL_ONE_STEP_REFERENCE = pd.DataFrame(comparison_rows)
display(EXTERNAL_ONE_STEP_REFERENCE)
EXTERNAL_ONE_STEP_REFERENCE.to_csv(
    OUTPUT_DIR/"TRACE_external_one_step_reference_only.csv", index=False,
)


## 8. Ablation Study: Context Profile and Zero-Handling Strategy

Using the same final model family, the notebook compares the validation-selected configuration with `forced hurdle`, `forced direct-all`, removal of the structural-zero rule, single-expert variants, and persistence baselines. In full mode, it also independently retrains a model without the weather-proxy family and a `full_context` model that restores weekly-memory features. Each retrained profile reselects its strategy on validation data and is then refitted on training plus validation data. The core ablations are therefore independently trained comparisons rather than post hoc output masks.


In [ ]:
def run_retrained_profile_ablation(site, full_result, packs, variant, keep_columns):
    train = subset_pack(packs["train"], keep_columns, "train")
    val = subset_pack(packs["val"], keep_columns, "val")
    test = subset_pack(packs["test"], keep_columns, "test")
    train_gate = learn_structural_gate(train)
    val_models = fit_fixed_experts(train, train_gate, full_result.params, 7200)
    val_models["direct_reg"] = fit_direct_all(train, train_gate, full_result.params["hist_reg"], SEED+7300)
    hurdle_decision, _ = select_decision(val, train_gate, predict_experts(val_models, val.X))
    temp_decision = {**hurdle_decision, "strategy": "hurdle"}
    val_components = predict_all_components(val_models, val.X, val.meta, train_gate, temp_decision)
    strategy, _, _, _, _ = choose_guarded_strategy(
        val, val_components["hurdle"], val_components["direct_all"],
    )
    del val_models
    gc.collect()

    final = combine_train_validation(site, train, val)
    final_gate = learn_structural_gate(final)
    final_models = fit_fixed_experts(final, final_gate, full_result.params, 7400)
    final_models["direct_reg"] = fit_direct_all(final, final_gate, full_result.params["hist_reg"], SEED+7500)
    decision = {**hurdle_decision, "strategy": strategy}
    components = predict_all_components(final_models, test.X, test.meta, final_gate, decision)
    pred = components["selected"]
    del final_models
    gc.collect()
    return pred, {**decision, "variant": variant}


ABLATION_ROWS = []
ABLATION_PREDICTIONS = {}
for site, result in RESULTS.items():
    p = result.predictions
    X = result.test_features
    full = p.pred_TRACE.to_numpy()
    variants = {
        "TRACE selected profile + strategy": full,
        "Forced hurdle strategy": p.pred_hurdle.to_numpy(),
        "Forced direct-all strategy": p.pred_direct_all.to_numpy(),
        "Previous-day target profile": np.maximum(X["pv_target_lag24"].to_numpy(), 0.0),
    }
    if "pv_target_lag168" in X:
        variants["Previous-week target profile"] = np.maximum(X["pv_target_lag168"].to_numpy(), 0.0)
    else:
        variants["Previous-week target profile"] = np.maximum(
            PACKS[site]["test"].X["pv_target_lag168"].to_numpy(), 0.0
        )

    empty_gate = {"structural_zero_keys": []}
    no_structural = predict_all_components(
        result.models, X, p, empty_gate, result.decision,
    )["selected"]
    variants["w/o structural-zero gate"] = no_structural

    extra_decision = {**result.decision, "strategy": "hurdle", "weight_extra_classifier": 1.0,
                      "weight_extra_regressor": 1.0}
    hist_decision = {**result.decision, "strategy": "hurdle", "weight_extra_classifier": 0.0,
                     "weight_extra_regressor": 0.0}
    variants["Extra Trees hurdle only"] = predict_all_components(
        result.models, X, p, result.final_gate, extra_decision,
    )["hurdle"]
    variants["Histogram boosting hurdle only"] = predict_all_components(
        result.models, X, p, result.final_gate, hist_decision,
    )["hurdle"]

    if CFG.run_retrained_ablation:
        selected_cols = X.columns.tolist()
        no_proxy_cols = [
            c for c in selected_cols
            if not c.startswith("wx_proxy_") and not c.startswith("interaction_proxy_")
        ]
        profile_variants = {"w/o weather-proxy family (retrained)": no_proxy_cols}
        if result.decision["feature_profile"] != "full_context":
            profile_variants["Full weekly-context profile (retrained)"] = PACKS[site]["train"].X.columns.tolist()
        for variant, cols in profile_variants.items():
            pred, audit = run_retrained_profile_ablation(site, result, PACKS[site], variant, cols)
            variants[variant] = pred
            print(site, audit)

    ABLATION_PREDICTIONS[site] = variants
    for variant, pred in variants.items():
        ABLATION_ROWS.append({"site": site, "variant": variant, **regression_metrics(p.actual, pred)})

ABLATION = pd.DataFrame(ABLATION_ROWS)
display(ABLATION.sort_values(["site", "RMSE"]))
ABLATION.to_csv(OUTPUT_DIR/"TRACE_ablation.csv", index=False)


## 9. Publication-Quality Visualization and Strategy-Aware XAI

The diagnostic figures show a complete 24-hour trajectory for one forecast origin, horizon-wise errors, and predicted activity probabilities. The explainability analysis includes:

1. global importance for the hurdle activity classifier and positive-magnitude regressor;
2. feature-family permutation importance for the **complete selected strategy**;
3. horizon-specific permutation-importance heatmaps; and
4. local TreeSHAP explanations for the selected strategy.

When direct-all is selected, local SHAP explains the direct regressor. When the hurdle strategy is selected, local SHAP separately explains activity and positive magnitude. Figures are saved at 600 dpi by default.


In [ ]:
# ============================================================
# Plot font-size configuration
# ============================================================
FONT_SIZE = {
    "base": 14,
    "title": 17,
    "suptitle": 20,
    "label": 15,
    "tick": 13,
    "legend": 13,
    "annotation": 12,
}

# Apply these font settings to all subsequent Matplotlib and Seaborn figures.
plt.rcParams.update({
    "font.size": FONT_SIZE["base"],
    "axes.titlesize": FONT_SIZE["title"],
    "axes.labelsize": FONT_SIZE["label"],
    "xtick.labelsize": FONT_SIZE["tick"],
    "ytick.labelsize": FONT_SIZE["tick"],
    "legend.fontsize": FONT_SIZE["legend"],
    "figure.titlesize": FONT_SIZE["suptitle"],
})


def apply_axis_font(ax, title_size=None):
    """Apply consistent title, label, tick, and legend font sizes to one axis."""
    title_size = title_size or FONT_SIZE["title"]

    ax.title.set_fontsize(title_size)
    ax.xaxis.label.set_fontsize(FONT_SIZE["label"])
    ax.yaxis.label.set_fontsize(FONT_SIZE["label"])
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=FONT_SIZE["tick"],
    )

    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(FONT_SIZE["legend"])

        legend_title = legend.get_title()
        if legend_title is not None:
            legend_title.set_fontsize(FONT_SIZE["legend"])


def plot_diagnostics(site: str, result: Result):
    p = result.predictions
    origin_energy = p.groupby("forecast_origin").actual.sum()
    chosen_origin = origin_energy.idxmax()
    g = p[p.forecast_origin == chosen_origin].sort_values("horizon")
    hm = HORIZON_METRICS[HORIZON_METRICS.site == site]

    # Slightly enlarge the figure to accommodate the larger fonts.
    fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))

    axes[0].plot(
        g.horizon,
        g.actual,
        "o-",
        label="Actual",
        color="#17202a",
        lw=2.0,
        markersize=6,
    )
    axes[0].plot(
        g.horizon,
        g.pred_TRACE,
        "o--",
        label="TRACE",
        color="#c0392b",
        lw=1.8,
        markersize=6,
    )
    axes[0].set_xlabel(
        "Forecast horizon (hour)",
        fontsize=FONT_SIZE["label"],
    )
    axes[0].set_ylabel(
        "Solar power (kW)",
        fontsize=FONT_SIZE["label"],
    )
    axes[0].set_title(
        f"24-step trajectory\norigin={pd.Timestamp(chosen_origin)}",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )
    axes[0].set_xticks(range(1, 25, 2))
    axes[0].legend(fontsize=FONT_SIZE["legend"])

    axes[1].plot(
        hm.horizon,
        hm.RMSE,
        "o-",
        label="RMSE",
        color="#2874a6",
        lw=2.0,
        markersize=6,
    )
    axes[1].plot(
        hm.horizon,
        hm.MAE,
        "o-",
        label="MAE",
        color="#d68910",
        lw=2.0,
        markersize=6,
    )
    axes[1].set_xlabel(
        "Forecast horizon",
        fontsize=FONT_SIZE["label"],
    )
    axes[1].set_ylabel(
        "Error (kW)",
        fontsize=FONT_SIZE["label"],
    )
    axes[1].set_title(
        "Error degradation by horizon",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )
    axes[1].set_xticks(range(1, 25))
    axes[1].legend(fontsize=FONT_SIZE["legend"])

    sample = p.sample(min(5000, len(p)), random_state=SEED)
    axes[2].scatter(
        sample.target_solar_elevation,
        sample.active_probability,
        c=sample.actual_active.astype(int),
        cmap="coolwarm",
        s=10,
        alpha=0.30,
    )
    axes[2].axhline(
        result.decision["threshold"],
        color="black",
        ls="--",
        lw=1.3,
    )
    axes[2].set_xlabel(
        "Target solar-elevation proxy",
        fontsize=FONT_SIZE["label"],
    )
    axes[2].set_ylabel(
        "Predicted activity probability",
        fontsize=FONT_SIZE["label"],
    )
    axes[2].set_title(
        "Zero/positive activity gate",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )

    for ax in axes:
        apply_axis_font(ax)

    fig.suptitle(
        f"{site}: TRACE operational diagnostics",
        fontsize=FONT_SIZE["suptitle"],
        fontweight="bold",
        y=1.02,
    )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{site}_TRACE_diagnostics.png",
        dpi=CFG.figure_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()


def feature_family(name: str):
    if name.startswith("pv_"):
        return "PV history"
    if name.startswith("wx_origin_"):
        return "Observed weather history"
    if name.startswith("wx_proxy_"):
        return "Weather forecast proxy"
    if name.startswith("target_") or name.startswith("lead_"):
        return "Known future context"
    if name.startswith("interaction_"):
        return "Physical interaction"
    return "Other"


def plot_global_xai(site: str, result: Result):
    names = result.test_features.columns.to_numpy()
    cls_imp = result.models["extra_cls"].feature_importances_
    reg_imp = result.models["extra_reg"].feature_importances_

    frame = pd.DataFrame({
        "feature": names,
        "activity_importance": cls_imp,
        "magnitude_importance": reg_imp,
    })
    frame["family"] = frame.feature.map(feature_family)

    # Enlarge the figure because the y-axis contains many feature names.
    fig, axes = plt.subplots(1, 2, figsize=(21, 8.5))

    a = (
        frame.nlargest(20, "activity_importance")
        .sort_values("activity_importance")
    )
    b = (
        frame.nlargest(20, "magnitude_importance")
        .sort_values("magnitude_importance")
    )

    axes[0].barh(
        a.feature,
        a.activity_importance,
        color="#c0392b",
    )
    axes[0].set_title(
        "Activity classifier: global importance",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )
    axes[0].set_xlabel(
        "Extra Trees importance",
        fontsize=FONT_SIZE["label"],
    )

    axes[1].barh(
        b.feature,
        b.magnitude_importance,
        color="#2471a3",
    )
    axes[1].set_title(
        "Positive magnitude regressor: global importance",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )
    axes[1].set_xlabel(
        "Extra Trees importance",
        fontsize=FONT_SIZE["label"],
    )

    for ax in axes:
        apply_axis_font(ax)
        ax.tick_params(axis="y", labelsize=FONT_SIZE["tick"])

    fig.suptitle(
        f"{site}: two-stage XAI",
        fontsize=FONT_SIZE["suptitle"],
        fontweight="bold",
        y=1.01,
    )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{site}_TRACE_two_stage_XAI.png",
        dpi=CFG.figure_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()

    frame.to_csv(
        OUTPUT_DIR / f"{site}_TRACE_feature_importance.csv",
        index=False,
    )

    family = (
        frame.groupby("family", as_index=False)[
            ["activity_importance", "magnitude_importance"]
        ]
        .sum()
    )

    display(
        family.sort_values(
            "magnitude_importance",
            ascending=False,
        )
    )

    return frame, family


def predict_trace_from_features(
    result: Result,
    X: pd.DataFrame,
    meta: pd.DataFrame,
):
    return predict_all_components(
        result.models,
        X,
        meta,
        result.final_gate,
        result.decision,
    )["selected"]


def family_permutation_xai(
    site: str,
    result: Result,
    max_rows=None,
    repeats=5,
):
    X = result.test_features.reset_index(drop=True)
    meta = result.predictions.reset_index(drop=True)

    n = min(max_rows or CFG.xai_rows, len(X))
    rng = np.random.default_rng(SEED)

    # Randomly sample test rows while preserving reproducibility with a fixed seed.
    selected = np.sort(
        rng.choice(
            len(X),
            size=n,
            replace=False,
        )
    )

    Xs = X.iloc[selected].copy()
    ms = meta.iloc[selected].copy().reset_index(drop=True)
    ys = ms.actual.to_numpy()

    baseline = regression_metrics(
        ys,
        predict_trace_from_features(result, Xs, ms),
    )["RMSE"]

    families = pd.Series({
        c: feature_family(c)
        for c in X.columns
    })

    rows = []

    for family in sorted(families.unique()):
        cols = families.index[
            families == family
        ].tolist()

        increases = []

        for _ in range(repeats):
            perm = rng.permutation(n)
            Xp = Xs.copy()

            Xp.loc[:, cols] = (
                Xs.iloc[perm][cols].to_numpy()
            )

            score = regression_metrics(
                ys,
                predict_trace_from_features(
                    result,
                    Xp,
                    ms,
                ),
            )["RMSE"]

            increases.append(score - baseline)

        rows.append({
            "site": site,
            "family": family,
            "features": len(cols),
            "baseline_RMSE": baseline,
            "RMSE_increase_mean": np.mean(increases),
            "RMSE_increase_std": (
                np.std(increases, ddof=1)
                if len(increases) > 1
                else 0.0
            ),
        })

    table = (
        pd.DataFrame(rows)
        .sort_values(
            "RMSE_increase_mean",
            ascending=False,
        )
    )

    table.to_csv(
        OUTPUT_DIR /
        f"{site}_TRACE_family_permutation_XAI.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(11, 6.5))

    plot = table.sort_values("RMSE_increase_mean")

    ax.barh(
        plot.family,
        plot.RMSE_increase_mean,
        xerr=plot.RMSE_increase_std,
        color="#2874a6",
        alpha=0.9,
        capsize=4,
    )
    ax.axvline(
        0,
        color="black",
        lw=1.0,
    )
    ax.set_xlabel(
        "RMSE increase after grouped permutation (kW)",
        fontsize=FONT_SIZE["label"],
    )
    ax.set_ylabel(
        "Feature family",
        fontsize=FONT_SIZE["label"],
    )
    ax.set_title(
        f"{site}: end-to-end family importance",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=12,
    )

    apply_axis_font(ax)

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR /
        f"{site}_TRACE_family_permutation_XAI.png",
        dpi=CFG.figure_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()

    return table


def horizon_family_permutation_xai(
    site: str,
    result: Result,
    per_horizon=400,
):
    X = result.test_features.reset_index(drop=True)
    meta = result.predictions.reset_index(drop=True)

    families = pd.Series({
        c: feature_family(c)
        for c in X.columns
    })

    rng = np.random.default_rng(SEED + 7)
    rows = []

    for h in range(1, CFG.horizon + 1):
        candidates = np.flatnonzero(
            meta.horizon.to_numpy() == h
        )

        idx = np.sort(
            rng.choice(
                candidates,
                size=min(per_horizon, len(candidates)),
                replace=False,
            )
        )

        Xs = X.iloc[idx].copy()
        ms = meta.iloc[idx].copy().reset_index(drop=True)
        ys = ms.actual.to_numpy()

        base = regression_metrics(
            ys,
            predict_trace_from_features(
                result,
                Xs,
                ms,
            ),
        )["RMSE"]

        for family in sorted(families.unique()):
            cols = families.index[
                families == family
            ].tolist()

            increases = []

            for _ in range(3):
                Xp = Xs.copy()
                perm = rng.permutation(len(Xs))

                Xp.loc[:, cols] = (
                    Xs.iloc[perm][cols].to_numpy()
                )

                permuted_rmse = regression_metrics(
                    ys,
                    predict_trace_from_features(
                        result,
                        Xp,
                        ms,
                    ),
                )["RMSE"]

                increases.append(
                    permuted_rmse - base
                )

            rows.append({
                "site": site,
                "horizon": h,
                "family": family,
                "RMSE_increase": float(
                    np.mean(increases)
                ),
            })

    table = pd.DataFrame(rows)

    matrix = table.pivot(
        index="family",
        columns="horizon",
        values="RMSE_increase",
    )

    fig, ax = plt.subplots(figsize=(18, 7))

    heatmap = sns.heatmap(
        matrix,
        cmap="rocket",
        center=0,
        ax=ax,
        cbar_kws={
            "label": "RMSE increase (kW)",
        },
    )

    ax.set_title(
        f"{site}: horizon-specific feature-family "
        "permutation importance",
        fontsize=FONT_SIZE["title"],
        fontweight="bold",
        pad=14,
    )
    ax.set_xlabel(
        "Forecast horizon",
        fontsize=FONT_SIZE["label"],
    )
    ax.set_ylabel(
        "Feature family",
        fontsize=FONT_SIZE["label"],
    )

    ax.tick_params(
        axis="x",
        labelsize=FONT_SIZE["tick"],
        rotation=0,
    )
    ax.tick_params(
        axis="y",
        labelsize=FONT_SIZE["tick"],
        rotation=0,
    )

    # Increase the heatmap colorbar label and tick sizes.
    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.tick_params(
        labelsize=FONT_SIZE["tick"],
    )
    colorbar.set_label(
        "RMSE increase (kW)",
        fontsize=FONT_SIZE["label"],
        labelpad=12,
    )

    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR /
        f"{site}_TRACE_horizon_family_XAI.png",
        dpi=CFG.figure_dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()

    table.to_csv(
        OUTPUT_DIR /
        f"{site}_TRACE_horizon_family_XAI.csv",
        index=False,
    )

    return table


def local_two_stage_shap(site: str, result: Result):
    try:
        import shap
    except ImportError:
        print(
            "Optional local SHAP skipped: "
            "Run `%pip install shap`, restart the kernel if needed, and call this function again."
        )
        return None

    meta = result.predictions.reset_index(drop=True)
    X = result.test_features.reset_index(drop=True)

    eligible = np.flatnonzero(
        meta.actual.to_numpy() > 0
    )

    sample_idx = int(
        eligible[
            np.argmax(
                meta.actual.to_numpy()[eligible]
            )
        ]
    )

    row = X.iloc[[sample_idx]]

    if result.decision["strategy"] == "direct_all":
        targets = [
            (
                "selected_direct_all",
                "direct_reg",
                None,
            )
        ]
    else:
        targets = [
            ("activity", "extra_cls", 1),
            ("positive_magnitude", "extra_reg", None),
        ]

    outputs = {}

    for stage, key, class_index in targets:
        explainer = shap.TreeExplainer(
            result.models[key]
        )
        exp = explainer(row)

        if exp.values.ndim == 3:
            exp = shap.Explanation(
                values=exp.values[:, :, class_index],
                base_values=np.asarray(
                    exp.base_values
                )[:, class_index],
                data=exp.data,
                feature_names=exp.feature_names,
            )

        # Enlarge the SHAP figure.
        plt.figure(figsize=(12, 8))

        shap.plots.waterfall(
            exp[0],
            max_display=18,
            show=False,
        )

        ax = plt.gca()

        ax.set_title(
            f"{site}: local {stage} explanation\n"
            f"target={meta.target_time.iloc[sample_idx]}",
            fontsize=FONT_SIZE["title"],
            fontweight="bold",
            pad=15,
        )

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=FONT_SIZE["tick"],
        )

        ax.xaxis.label.set_fontsize(
            FONT_SIZE["label"]
        )
        ax.yaxis.label.set_fontsize(
            FONT_SIZE["label"]
        )

        # Increase the font size of text objects created by SHAP.
        for text in ax.texts:
            text.set_fontsize(
                FONT_SIZE["annotation"]
            )

        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR /
            f"{site}_TRACE_local_SHAP_{stage}.png",
            dpi=CFG.figure_dpi,
            bbox_inches="tight",
            facecolor="white",
        )
        plt.show()

        outputs[stage] = exp

    return outputs


XAI = {}

for site, result in RESULTS.items():
    plot_diagnostics(site, result)

    if CFG.run_xai:
        XAI[site] = {
            "tree_importance": plot_global_xai(
                site,
                result,
            ),
            "family_permutation": family_permutation_xai(
                site,
                result,
            ),
            "horizon_family": horizon_family_permutation_xai(
                site,
                result,
            ),
            "local_shap": local_two_stage_shap(
                site,
                result,
            ),
        }

## 10. Statistical Validation: Date-Block Bootstrap, HAC-DM, Wilcoxon, and Holm Correction

Origin–horizon pairs overlap strongly, so the notebook does not treat individual rows as independent observations. It first averages losses across all 24 horizons and all forecast origins within each forecast-origin date. TRACE's standalone performance interval is estimated with a date-block bootstrap. Differences against each benchmark or ablation are evaluated using:

1. a paired Wilcoxon test on daily losses;
2. a HAC Diebold–Mariano-type test with a seven-day Newey–West long-run variance estimator; and
3. a date-block bootstrap confidence interval for the paired loss difference.

Within each site and loss family, multiple-comparison p-values are adjusted with Holm's procedure.

The loss difference is defined as `comparison loss − TRACE loss`, so positive values favor TRACE. `CLAIM_AUDIT` permits a superiority claim over TCFN only when TRACE has lower RMSE and MAE and both Holm-adjusted HAC p-values for absolute and squared loss are below 0.05.


In [ ]:
def origin_day_bootstrap(predictions: pd.DataFrame, repeats: int, seed: int):
    p = predictions.copy().reset_index(drop=True)
    p["origin_date"] = pd.to_datetime(p.forecast_origin).dt.date
    dates = np.array(sorted(p.origin_date.unique()), dtype=object)
    groups = {d: p.index[p.origin_date == d].to_numpy() for d in dates}
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(repeats):
        sampled = rng.choice(dates, size=len(dates), replace=True)
        idx = np.concatenate([groups[d] for d in sampled])
        draws.append(regression_metrics(p.actual.iloc[idx], p.pred_TRACE.iloc[idx]))
    dist = pd.DataFrame(draws)
    point = regression_metrics(p.actual, p.pred_TRACE)
    summary = []
    for metric in ("RMSE", "MAE", "R2"):
        summary.append({
            "metric": metric, "point": point[metric],
            "ci_2.5%": dist[metric].quantile(.025), "ci_97.5%": dist[metric].quantile(.975),
            "bootstrap_mean": dist[metric].mean(), "bootstrap_std": dist[metric].std(ddof=1),
        })
    return pd.DataFrame(summary), dist

BOOTSTRAP_ROWS = []
for site, result in RESULTS.items():
    summary, dist = origin_day_bootstrap(result.predictions, CFG.bootstrap_repeats, SEED)
    summary.insert(0, "site", site)
    BOOTSTRAP_ROWS.append(summary)
    dist.to_csv(OUTPUT_DIR/f"{site}_TRACE_bootstrap_distribution.csv", index=False)
BOOTSTRAP_CI = pd.concat(BOOTSTRAP_ROWS, ignore_index=True)
display(BOOTSTRAP_CI)
BOOTSTRAP_CI.to_csv(OUTPUT_DIR/"TRACE_bootstrap_CI.csv", index=False)

def holm_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    m = len(p); order = np.argsort(p); adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m-rank)*p[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted

def hac_dm_test(daily_difference, max_lag=7):
    """H0: E[comparison loss - TRACE loss] = 0; two-sided normal approximation."""
    d = np.asarray(daily_difference, dtype=float)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 10 or np.allclose(d, d[0]):
        return np.nan, np.nan
    centered = d-d.mean()
    gamma0 = np.dot(centered, centered)/n
    long_run = gamma0
    L = min(int(max_lag), n-2)
    for lag in range(1, L+1):
        gamma = np.dot(centered[lag:], centered[:-lag])/n
        long_run += 2*(1-lag/(L+1))*gamma
    se = np.sqrt(max(long_run, np.finfo(float).eps)/n)
    stat = d.mean()/se
    return float(stat), float(2*norm.sf(abs(stat)))

def paired_daily_loss_table(p, trace_pred, comparison_pred, loss_name):
    actual = p.actual.to_numpy(dtype=float)
    if loss_name == "absolute":
        trace_loss = np.abs(actual-trace_pred)
        comparison_loss = np.abs(actual-comparison_pred)
    elif loss_name == "squared":
        trace_loss = np.square(actual-trace_pred)
        comparison_loss = np.square(actual-comparison_pred)
    else:
        raise ValueError(loss_name)
    table = pd.DataFrame({
        "origin_date": pd.to_datetime(p.forecast_origin).dt.floor("D"),
        "trace_loss": trace_loss, "comparison_loss": comparison_loss,
    }).groupby("origin_date", as_index=False).mean(numeric_only=True)
    table["difference"] = table.comparison_loss-table.trace_loss
    return table

def compare_against_trace(site, comparison_type, name, pred, repeats, seed):
    p = RESULTS[site].predictions.reset_index(drop=True)
    trace = p.pred_TRACE.to_numpy(dtype=float)
    pred = np.asarray(pred, dtype=float)
    if len(pred) != len(trace):
        raise AssertionError(f"{site}/{name}: paired length mismatch")
    rows = []
    for loss_name in ("absolute", "squared"):
        day = paired_daily_loss_table(p, trace, pred, loss_name)
        d = day.difference.to_numpy()
        if np.allclose(d, 0.0):
            w_stat, w_p = np.nan, 1.0
        else:
            try:
                w = wilcoxon(d, zero_method="pratt", alternative="two-sided", method="auto")
                w_stat, w_p = float(w.statistic), float(w.pvalue)
            except ValueError:
                w_stat, w_p = np.nan, 1.0
        dm_stat, dm_p = hac_dm_test(d, max_lag=7)
        rng = np.random.default_rng(seed+sum(map(ord, name+loss_name)))
        boot = np.array([
            rng.choice(d, size=len(d), replace=True).mean() for _ in range(repeats)
        ])
        rows.append({
            "site": site, "comparison_type": comparison_type, "comparison": name,
            "loss": loss_name, "origin_days": len(day),
            "mean_loss_difference": float(d.mean()),
            "median_loss_difference": float(np.median(d)),
            "TRACE_win_rate": float(np.mean(d > 0)),
            "paired_standardized_effect": float(d.mean()/d.std(ddof=1)) if d.std(ddof=1) > 0 else np.nan,
            "difference_ci_2.5%": float(np.quantile(boot, .025)),
            "difference_ci_97.5%": float(np.quantile(boot, .975)),
            "wilcoxon_stat": w_stat, "wilcoxon_p_two_sided": w_p,
            "HAC_DM_stat": dm_stat, "HAC_DM_p_two_sided": dm_p,
        })
    return rows

statistical_rows = []
for site, preds in PAIR_PREDICTIONS.items():
    for name, pred in preds.items():
        if name != "TRACE":
            statistical_rows.extend(compare_against_trace(
                site, "benchmark", name, pred, CFG.bootstrap_repeats, SEED,
            ))
    for name, pred in ABLATION_PREDICTIONS[site].items():
        if name != "TRACE full hurdle":
            statistical_rows.extend(compare_against_trace(
                site, "ablation", name, pred, CFG.bootstrap_repeats, SEED+500,
            ))

PAIRED_TESTS = pd.DataFrame(statistical_rows)
if len(PAIRED_TESTS):
    PAIRED_TESTS["wilcoxon_p_holm"] = np.nan
    PAIRED_TESTS["HAC_DM_p_holm"] = np.nan
    for _, idx in PAIRED_TESTS.groupby(["site", "comparison_type", "loss"]).groups.items():
        idx = list(idx)
        PAIRED_TESTS.loc[idx, "wilcoxon_p_holm"] = holm_adjust(
            PAIRED_TESTS.loc[idx, "wilcoxon_p_two_sided"].fillna(1.0)
        )
        PAIRED_TESTS.loc[idx, "HAC_DM_p_holm"] = holm_adjust(
            PAIRED_TESTS.loc[idx, "HAC_DM_p_two_sided"].fillna(1.0)
        )
display(PAIRED_TESTS.sort_values(["site", "comparison_type", "loss", "HAC_DM_p_holm"]))
PAIRED_TESTS.to_csv(OUTPUT_DIR/"paired_day_block_statistical_tests.csv", index=False)

claim_rows = []
for site in RESULTS:
    overall = FAIR_BENCHMARK_OVERALL[FAIR_BENCHMARK_OVERALL.site == site].set_index("model")
    for tcfn_name in [m for m in ("TCFN", "TCFN-168") if m in overall.index]:
        trace_m = overall.loc["TRACE"]
        tcfn_m = overall.loc[tcfn_name]
        tests = PAIRED_TESTS[
            (PAIRED_TESTS.site == site) & (PAIRED_TESTS.comparison_type == "benchmark") &
            (PAIRED_TESTS.comparison == tcfn_name)
        ].set_index("loss")
        rmse_gain = 100*(tcfn_m.RMSE-trace_m.RMSE)/tcfn_m.RMSE
        mae_gain = 100*(tcfn_m.MAE-trace_m.MAE)/tcfn_m.MAE
        pass_metrics = (rmse_gain > 0) and (mae_gain > 0)
        pass_stats = (
            set(["absolute", "squared"]).issubset(tests.index) and
            (tests.loc[["absolute", "squared"], "HAC_DM_p_holm"] < .05).all() and
            (tests.loc[["absolute", "squared"], "mean_loss_difference"] > 0).all()
        )
        claim_rows.append({
            "site": site, "TCFN_comparator": tcfn_name,
            "RMSE_improvement_pct": rmse_gain, "MAE_improvement_pct": mae_gain,
            "absolute_HAC_DM_p_holm": tests.loc["absolute", "HAC_DM_p_holm"],
            "squared_HAC_DM_p_holm": tests.loc["squared", "HAC_DM_p_holm"],
            "lower_RMSE_and_MAE": pass_metrics,
            "significant_both_losses": bool(pass_stats),
            "superiority_claim_supported": bool(pass_metrics and pass_stats),
        })
CLAIM_AUDIT = pd.DataFrame(claim_rows)
display(CLAIM_AUDIT)
CLAIM_AUDIT.to_csv(OUTPUT_DIR/"TCFN_superiority_claim_audit.csv", index=False)
PRIMARY_CLAIM = CLAIM_AUDIT[CLAIM_AUDIT.TCFN_comparator == "TCFN"] if len(CLAIM_AUDIT) else CLAIM_AUDIT
if len(PRIMARY_CLAIM) == len(RESULTS) and PRIMARY_CLAIM.superiority_claim_supported.all():
    print("SUPPORTED: The predefined criteria support a TRACE superiority claim over TCFN.")
else:
    print("NOT YET SUPPORTED: At least one site–TCFN comparison does not satisfy the superiority criteria. Do not hide results or tune the method to the test set.")


## 11. Model Serialization and Hourly Operational Inference

Each saved bundle contains five fitted experts, the selected feature profile, the structural-zero rule, and the validation-locked dual-strategy decision. Given at least 168 consecutive hourly observations, `forecast_next_24` builds the complete causal feature set, aligns the selected columns, and returns the next 24 hourly forecasts. The output includes hurdle, direct-all, and final selected predictions to support operational auditing.


In [ ]:
def predict_from_bundle(X: pd.DataFrame, meta: pd.DataFrame, bundle):
    models = bundle["models"]
    decision = bundle["decision"]
    gate = bundle["structural_gate"]
    components = predict_all_components(models, X, meta, gate, decision)
    return components


def forecast_next_24(history: pd.DataFrame, site: str, bundle):
    history = history.copy().sort_index()
    if not isinstance(history.index, pd.DatetimeIndex):
        raise TypeError("history must have an hourly DatetimeIndex")
    required = ["Solar_Power", *OPERATIONAL_WEATHER]
    missing = sorted(set(required)-set(history.columns))
    if missing:
        raise ValueError(f"missing live columns: {missing}")
    if len(history) < CFG.history_hours:
        raise ValueError(f"at least {CFG.history_hours} history rows are required")
    history = history.iloc[-CFG.history_hours:].copy()
    expected = pd.date_range(history.index.min(), history.index.max(), freq="h")
    if len(expected) != len(history) or not history.index.equals(expected):
        raise ValueError("the latest 168-hour history must be continuous")
    if history[required].isna().any().any():
        raise ValueError("live history contains missing values")

    future = pd.date_range(history.index[-1]+pd.Timedelta(hours=1), periods=CFG.horizon, freq="h")
    extended = history.reindex(history.index.append(future))
    extended["observed_row"] = False
    extended.loc[history.index, "observed_row"] = True
    origin_pos = np.array([CFG.history_hours-1], dtype=int)
    X, meta, _ = assemble_origin_horizon_features(extended, site, origin_pos)
    X = X[bundle["feature_columns"]]
    components = predict_from_bundle(X, meta, bundle)
    out = meta.copy()
    out["active_probability"] = components["probability"]
    out["predicted_positive_magnitude"] = components["magnitude"]
    out["pred_hurdle"] = components["hurdle"]
    out["pred_direct_all"] = components["direct_all"]
    out["pred_TRACE"] = components["selected"]
    out["selected_strategy"] = bundle["decision"]["strategy"]
    return out


FINAL_ROWS = []
for site, result in RESULTS.items():
    bundle = {
        "method": "TRACE guarded dual strategy",
        "history_hours": CFG.history_hours,
        "forecast_horizon": CFG.horizon,
        "models": result.models,
        "params": result.params,
        "decision": result.decision,
        "structural_gate": result.final_gate,
        "feature_profile": result.decision["feature_profile"],
        "feature_columns": result.test_features.columns.tolist(),
        "operational_weather": OPERATIONAL_WEATHER,
    }
    bundle_path = OUTPUT_DIR/f"{site}_TRACE_bundle.joblib"
    joblib.dump(bundle, bundle_path, compress=3)
    reloaded_bundle = joblib.load(bundle_path)
    live_cutoff = pd.Timestamp(CFG.test_start)-pd.Timedelta(hours=1)
    live_history = DATA[site].loc[:live_cutoff, ["Solar_Power", *OPERATIONAL_WEATHER]].tail(CFG.history_hours)
    live_check = forecast_next_24(live_history, site, reloaded_bundle)
    assert len(live_check) == CFG.horizon
    assert live_check.pred_TRACE.notna().all()
    assert (live_check.target_time > live_check.forecast_origin).all()
    print(site, "reloaded live bundle audit passed:", reloaded_bundle["decision"]["strategy"])
    result.predictions.to_csv(OUTPUT_DIR/f"{site}_TRACE_test_predictions.csv", index=False)
    overall = regression_metrics(result.predictions.actual, result.predictions.pred_TRACE)
    FINAL_ROWS.append({
        "site": site, "feature_profile": result.decision["feature_profile"],
        "strategy": result.decision["strategy"],
        "test_origins": result.predictions.forecast_origin.nunique(),
        "origin_horizon_pairs": len(result.predictions), **overall,
        "refit_seconds": result.refit_seconds,
        "weight_extra_classifier": result.decision["weight_extra_classifier"],
        "weight_extra_regressor": result.decision["weight_extra_regressor"],
        "threshold": result.decision["threshold"], "mode": result.decision["mode"],
    })
FINAL_RESULTS = pd.DataFrame(FINAL_ROWS)
display(FINAL_RESULTS)
FINAL_RESULTS.to_csv(OUTPUT_DIR/"TRACE_final_results.csv", index=False)
with open(OUTPUT_DIR/"TRACE_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)
EXPERIMENT_MANIFEST = {
    "split_unit": "calendar month",
    "train": "2015-01-01 00:00 through 2017-04-30 23:00 (28 months; 50%)",
    "validation": "2017-05-01 00:00 through 2018-06-30 23:00 (14 months; 25%)",
    "test": "2018-07-01 00:00 through 2019-08-31 23:00 (14 months; 25%)",
    "forecast_task": "rolling direct t+1...t+24, updated hourly",
    "primary_metric_scope": "all zeros and all origin-horizon pairs included",
    "tcfn_repository": "https://github.com/johnnyone89/TCFN4PVForecasting",
    "tcfn_commit_verified": "bdb5aa1c6fd9cfe1bb84704930567201e074cf94",
    "selection_rule": "validation-only context profile plus guarded dual strategy; train+validation refit; test once",
    "direct_switch_rule": "direct-all requires at least 2% lower validation RMSE and non-inferior validation MAE",
    "future_observed_weather_used": False,
    "confirmatory_status": "exploratory redesign on an already observed test; external/future frozen-model confirmation required",
}
with open(OUTPUT_DIR/"experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_MANIFEST, f, ensure_ascii=False, indent=2)
print("saved to:", OUTPUT_DIR)


## Required Reporting and Deployment Notes

- Report **overall direct 24-step metrics with observed zeros included** as the primary results.
- The calendar split is 28/14/14 months (50/25/25), and test targets span 1 July 2018 through 31 August 2019.
- Report the site-specific feature-profile and strategy-selection rationale stored in `validation_profile_strategy_audit.csv`.
- Select direct-all only when validation RMSE improves by at least 2% relative to hurdle and validation MAE does not worsen.
- Clearly label solar-eligible and actual-positive results as supplementary or diagnostic analyses.
- With `FAST_MODE=False`, each site's test set should contain 10,225 hourly forecast origins and 245,400 origin–horizon pairs.
- The primary fair comparison is the direct 24-step TCFN and same-target benchmark table retrained in this notebook. The one-step values from TCFN Table 3 are external references only.
- Before claiming superiority over TCFN, confirm that `superiority_claim_supported=True` for both sites in `TCFN_superiority_claim_audit.csv`.
- Results generated with `FAST_MODE=True` or `TRACE_SKIP_DEEP=1` are smoke-test outputs and must not be reported as paper results.
- The datasets do not contain issue-time NWP forecasts; the model therefore uses a weather proxy derived from past observations. Supplying actual future weather would constitute an oracle analysis.
- The dual-strategy candidates were designed after diagnosing earlier results on the same test period. Improvements on July 2018–August 2019 are therefore developmental evidence, not confirmatory evidence of generalization. Validate a frozen model on a later period or an external site before making confirmatory claims.
- In live operation, call `forecast_next_24` again after each new hourly PV and weather observation becomes available.
